In [ ]:

# =============================================================
# NETGUARD CURRENT ATTACK DETECTION
# CELL 1 — ENVIRONMENT AND SAVED MODEL ARTIFACT AUDIT
#
# READ-ONLY
# No training
# No TEST data access
# No checkpoint modification
# =============================================================

from google.colab import drive
from pathlib import Path
import sys
import os

drive.mount("/content/drive")

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

print("=" * 75)
print("NETGUARD — CURRENT ATTACK DETECTION")
print("CELL 1: ENVIRONMENT AND MODEL ARTIFACT AUDIT")
print("=" * 75)

print("\nPython version:", sys.version.split()[0])
print("Project folder:", ROOT)
print("Project exists:", ROOT.exists())

if not ROOT.exists():
    raise FileNotFoundError(
        f"Project directory not found: {ROOT}"
    )

print("\nCURRENT ATTACK MODEL ARTIFACT SEARCH")
print("-" * 75)

# Search only model-like files by their filenames.
# Never open or read evaluation/TEST datasets.

MODEL_SUFFIXES = {
    ".json", ".joblib", ".pkl", ".pickle",
    ".pt", ".pth", ".h5", ".keras", ".onnx"
}

KEYWORDS = {
    "unsw", "attack_type", "classifier",
    "current", "xgboost", "multiclass",
    "binary", "preprocessing"
}

matches = []
checked_directories = 0

for current_dir, dirs, files in os.walk(ROOT):

    relative = Path(current_dir).relative_to(ROOT)

    if len(relative.parts) >= 8:
        dirs[:] = []
        continue

    # Exclude official TEST directories entirely.

    dirs[:] = [
        d for d in dirs
        if d.lower() not in {
            "test", "testing",
            "official_test", ".git",
            "node_modules", "__pycache__"
        }
    ]

    checked_directories += 1

    for filename in files:

        path = Path(current_dir) / filename

        if path.suffix.lower() not in MODEL_SUFFIXES:
            continue

        name = filename.lower()

        if not any(
            keyword in name
            or keyword in str(relative).lower()
            for keyword in KEYWORDS
        ):
            continue

        # Exclude any file with TEST in its name.
        if "test" in name:
            continue

        matches.append(path)

print("Directories inspected:", checked_directories)
print("Matching artifact files:", len(matches))

for path in sorted(matches)[:100]:
    print(
        " -",
        path.relative_to(ROOT)
    )

if len(matches) > 100:
    print("Additional matches omitted.")

print("\nEXPECTED CURRENT ATTACK ARTIFACTS")
print("-" * 75)

expected = [
    ROOT / "model/results/unsw_xgboost/"
           "unsw_attack_type_xgboost.json",

    ROOT / "model/results/unsw_xgboost/"
           "unsw_preprocessing.json",
]

for path in expected:

    print(
        "FOUND  " if path.is_file() else "MISSING",
        path.relative_to(ROOT)
    )

print("\n" + "=" * 75)
print("CELL 1 COMPLETE")
print("Models trained: NO")
print("Official TEST accessed: NO")
print("Files modified: NO")
print("=" * 75)


In [ ]:

# ================================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 2: ORIGINAL UNSW CLASSIFIER DISCOVERY
#
# READ-ONLY INVENTORY
# No dataset loading
# No official TEST access
# No model training
# No file writes
# ================================================================

from pathlib import Path
import os

ROOT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

if not ROOT.is_dir():
    raise FileNotFoundError(ROOT)

print("=" * 85)
print("CELL 2 — ORIGINAL CURRENT ATTACK CLASSIFIER DISCOVERY")
print("=" * 85)

# ------------------------------------------------
# 1. Exclude the new synthetic forecasting pipeline.
# ------------------------------------------------

EXCLUDE_DIRS = {
    ".git",
    "node_modules",
    "__pycache__",
    ".ipynb_checkpoints",
    "test",
    "testing",
    "official_test"
}

MODEL_EXTENSIONS = {
    ".json",
    ".joblib",
    ".pkl",
    ".pickle",
    ".ubj",
    ".bin",
    ".model",
    ".pt",
    ".pth",
    ".h5",
    ".keras",
    ".onnx"
}

SOURCE_EXTENSIONS = {
    ".py",
    ".ipynb"
}

ARTIFACT_KEYWORDS = (
    "unsw",
    "attack_type",
    "attack_cat",
    "classification",
    "classifier",
    "multiclass",
    "current",
    "xgboost",
    "preprocessing",
    "encoder"
)

notebooks = []
scripts = []
model_candidates = []
relevant_directories = []

scanned_dirs = 0

for current_dir, dirs, files in os.walk(ROOT):

    current_path = Path(current_dir)

    relative_dir = current_path.relative_to(ROOT)

    # Do not enter synthetic forecasting model trees,
    # TEST folders, or cache/build folders.
    dirs[:] = [
        d for d in dirs
        if d.lower() not in EXCLUDE_DIRS
        and not d.startswith("Synthetic_NAF")
        and not d.startswith("Synthetic_NAF_v")
    ]

    # Also avoid source trees already inside synthetic folders.
    if any(
        part.startswith("Synthetic_NAF")
        for part in relative_dir.parts
    ):
        dirs[:] = []
        continue

    scanned_dirs += 1

    folder_text = str(relative_dir).lower()

    if any(
        term in folder_text
        for term in ("unsw", "classifier", "current")
    ):
        relevant_directories.append(relative_dir)

    for filename in files:

        path = current_path / filename

        name = filename.lower()
        relative = path.relative_to(ROOT)

        # Never inspect files that explicitly identify
        # themselves as TEST artifacts.
        if any(
            "test" in part.lower()
            for part in relative.parts
        ):
            continue

        suffix = path.suffix.lower()

        if suffix == ".ipynb":
            notebooks.append(relative)

        elif suffix == ".py":
            scripts.append(relative)

        if suffix not in MODEL_EXTENSIONS:
            continue

        full_name = str(relative).lower()

        if not any(
            keyword in full_name
            for keyword in ARTIFACT_KEYWORDS
        ):
            continue

        try:
            size_kb = round(
                path.stat().st_size / 1024, 1
            )
        except OSError:
            size_kb = None

        model_candidates.append({
            "path": str(relative),
            "size_kb": size_kb
        })

# ------------------------------------------------
# 2. Print potential original classification files.
# ------------------------------------------------

print("\nDIRECTORIES SCANNED:", scanned_dirs)

print("\n" + "-" * 85)
print("A. CURRENT CLASSIFICATION ARTIFACT CANDIDATES")
print("-" * 85)

if model_candidates:

    for item in sorted(
        model_candidates,
        key=lambda x: x["path"]
    )[:100]:

        print(
            f"{item['size_kb']} KB | {item['path']}"
        )

else:
    print("No matching saved model artifacts found.")

# ------------------------------------------------
# 3. List notebooks without opening them.
# ------------------------------------------------

print("\n" + "-" * 85)
print("B. PROJECT NOTEBOOKS")
print("-" * 85)

print("Notebook count:", len(notebooks))

for notebook in sorted(notebooks)[:100]:
    print(" -", notebook)

# ------------------------------------------------
# 4. Find potentially relevant Python scripts.
# ------------------------------------------------

print("\n" + "-" * 85)
print("C. UNSW / CURRENT-ATTACK PYTHON SCRIPTS")
print("-" * 85)

relevant_scripts = [
    path
    for path in scripts
    if any(
        keyword in str(path).lower()
        for keyword in (
            "unsw",
            "attack",
            "train",
            "classif",
            "predict",
            "model"
        )
    )
]

for script in sorted(relevant_scripts)[:100]:
    print(" -", script)

if not relevant_scripts:
    print("No relevant Python scripts found.")

# ------------------------------------------------
# 5. List related directories.
# ------------------------------------------------

print("\n" + "-" * 85)
print("D. UNSW / CURRENT CLASSIFICATION DIRECTORIES")
print("-" * 85)

for directory in sorted(
    set(relevant_directories),
    key=str
)[:80]:

    print(" -", directory)

# ------------------------------------------------
# 6. Summary
# ------------------------------------------------

print("\n" + "=" * 85)
print("CELL 2 — DISCOVERY COMPLETE")
print("=" * 85)

print("Candidate artifact files:", len(model_candidates))
print("Project notebooks:", len(notebooks))
print("Relevant scripts:", len(relevant_scripts))

print("\nDatasets opened: NO")
print("Official TEST accessed: NO")
print("Models trained: NO")
print("Files modified: NO")

print("=" * 85)


In [ ]:

# ================================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 3: SAFE TRAIN/VALIDATION DATASET DISCOVERY
#
# READ-ONLY
# NO OFFICIAL TEST ACCESS
# NO MODEL TRAINING
# NO FILE WRITES
# ================================================================

from pathlib import Path
import os

MYDRIVE = Path("/content/drive/MyDrive")
ROOT = MYDRIVE / "network-attack-forecasting"

print("=" * 85)
print("CELL 3 — CURRENT ATTACK DATASET DISCOVERY")
print("=" * 85)

assert ROOT.is_dir(), "Project folder missing."

DATA_EXTENSIONS = {
    ".csv", ".parquet", ".feather"
}

EXCLUDED_DIRECTORIES = {
    "node_modules",
    ".git",
    "__pycache__",
    ".ipynb_checkpoints"
}

# Do not enter the completed synthetic forecasting
# dataset/model trees.
EXCLUDED_PREFIXES = (
    "Synthetic_NAF",
)

TRAIN_KEYWORDS = (
    "train",
    "training"
)

VAL_KEYWORDS = (
    "validation",
    "valid",
    "val_set"
)

UNSW_KEYWORDS = (
    "unsw",
    "nb15"
)

train_candidates = []
validation_candidates = []
unsw_other_candidates = []

scanned_directories = 0

# Search the project and the top levels of MyDrive
# for possible original classification datasets.

for current_dir, dirs, files in os.walk(MYDRIVE):

    current_path = Path(current_dir)
    relative = current_path.relative_to(MYDRIVE)

    depth = len(relative.parts)

    # Keep the traversal bounded.
    if depth >= 9:
        dirs[:] = []
        continue

    # Outside our project, only search shallow folders.
    in_project = (
        current_path == ROOT
        or ROOT in current_path.parents
    )

    if not in_project and depth >= 3:
        dirs[:] = []
        continue

    # Never enter TEST or synthetic model directories.
    dirs[:] = [
        d for d in dirs
        if d.lower() not in EXCLUDED_DIRECTORIES
        and "test" not in d.lower()
        and not any(
            d.startswith(prefix)
            for prefix in EXCLUDED_PREFIXES
        )
    ]

    scanned_directories += 1

    for name in files:

        path = current_path / name
        filename = name.lower()

        if path.suffix.lower() not in DATA_EXTENSIONS:
            continue

        # Explicitly exclude TEST-named files.
        if "test" in filename:
            continue

        relative_text = str(
            path.relative_to(MYDRIVE)
        ).lower()

        if not any(
            term in relative_text
            for term in UNSW_KEYWORDS
        ):
            continue

        if any(
            term in filename
            for term in TRAIN_KEYWORDS
        ):
            train_candidates.append(path)

        elif any(
            term in filename
            for term in VAL_KEYWORDS
        ):
            validation_candidates.append(path)

        else:
            unsw_other_candidates.append(path)


def display_paths(title, paths, limit=30):

    print("\n" + "-" * 85)
    print(title)
    print("-" * 85)

    print("Files found:", len(paths))

    for path in sorted(paths)[:limit]:

        size_mb = round(
            path.stat().st_size / (1024 ** 2),
            2
        )

        print(
            f"{size_mb:9.2f} MB | "
            f"{path.relative_to(MYDRIVE)}"
        )

    if len(paths) > limit:
        print(
            f"... {len(paths) - limit} additional files"
        )


display_paths(
    "A. POSSIBLE TRAIN DATASETS",
    train_candidates
)

display_paths(
    "B. POSSIBLE VALIDATION DATASETS",
    validation_candidates
)

display_paths(
    "C. OTHER UNSW FILES (NOT AUTOMATICALLY APPROVED)",
    unsw_other_candidates,
    limit=20
)

# ------------------------------------------------
# Inspect ONLY directory names and filenames for
# the existing raw UNSW collection.
# No dataset contents are accessed.
# ------------------------------------------------

RAW_UNSW = (
    ROOT
    / "Dataset_Audit_v2"
    / "raw"
    / "UNSW_NB15"
)

print("\n" + "-" * 85)
print("D. ORIGINAL RAW UNSW DIRECTORY")
print("-" * 85)

print("Exists:", RAW_UNSW.is_dir())

if RAW_UNSW.is_dir():

    for entry in sorted(RAW_UNSW.iterdir())[:30]:

        print(
            "[DIR]" if entry.is_dir() else "[FILE]",
            entry.name
        )

print("\n" + "=" * 85)
print("CELL 3 COMPLETE")
print("=" * 85)

print("Directories scanned:", scanned_directories)
print("TRAIN candidates:", len(train_candidates))
print("VALIDATION candidates:", len(validation_candidates))

print("\nDataset contents opened: NO")
print("Official TEST accessed: NO")
print("Models trained: NO")
print("Files modified: NO")
print("=" * 85)


In [ ]:

# ================================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 4: UNSW RAW SCHEMA AND ORIGINAL TRAIN FILE AUDIT
#
# READ ONLY
# NO TRAINING
# NO OFFICIAL TEST ACCESS
# NO FILE WRITES
# ================================================================

from pathlib import Path
import csv
import os
import pandas as pd

MYDRIVE = Path("/content/drive/MyDrive")

ROOT = MYDRIVE / "network-attack-forecasting"

RAW = (
    ROOT
    / "Dataset_Audit_v2"
    / "raw"
    / "UNSW_NB15"
)

print("=" * 85)
print("CELL 4 — UNSW CURRENT ATTACK DATA AUDIT")
print("=" * 85)

assert RAW.is_dir(), f"Missing directory: {RAW}"

# ------------------------------------------------------------
# A. READ FEATURE DEFINITIONS ONLY
# ------------------------------------------------------------

feature_file = RAW / "NUSW-NB15_features.csv"

print("\nA. OFFICIAL RAW FEATURE DEFINITIONS")
print("-" * 85)

if feature_file.is_file():

    features = pd.read_csv(
        feature_file,
        encoding="cp1252"
    )

    print("Feature metadata shape:", features.shape)
    print("Metadata columns:", features.columns.tolist())

    print("\nFirst eight feature definitions:")
    print(features.head(8).to_string(index=False))

else:

    print("Feature-definition file not found.")

# ------------------------------------------------------------
# B. INSPECT RAW CSV RECORD STRUCTURE
#
# UNSW-NB15_1 ... _4 can be headerless.
# Never assume first record is a header.
# Do not read the full datasets.
# ------------------------------------------------------------

print("\nB. RAW UNSW FILE STRUCTURE")
print("-" * 85)

for number in range(1, 5):

    path = RAW / f"UNSW-NB15_{number}.csv"

    if not path.is_file():

        print("MISSING:", path.name)
        continue

    with path.open(
        "r",
        encoding="cp1252",
        errors="replace",
        newline=""
    ) as handle:

        reader = csv.reader(handle)

        first_record = next(reader, None)

    column_count = (
        len(first_record)
        if first_record is not None
        else 0
    )

    print(
        f"{path.name}: "
        f"{path.stat().st_size / (1024 ** 2):.2f} MB, "
        f"{column_count} columns in first record"
    )

# ------------------------------------------------------------
# C. SEARCH MYDRIVE FOR AN ORIGINAL TRAIN-ONLY CSV
#
# Filenames only; never open TEST files.
# No synthetic-forecasting directories.
# ------------------------------------------------------------

print("\nC. ORIGINAL CLASSIFIER TRAINING CSV SEARCH")
print("-" * 85)

expected_names = {
    "unsw_nb15_training-set.csv",
    "unsw-nb15_training-set.csv",
    "unsw_nb15_train.csv",
    "unsw-nb15_train.csv",
    "unsw_training.csv",
}

found = []

skip_dirs = {
    ".git",
    "node_modules",
    "__pycache__",
    ".ipynb_checkpoints",
}

for folder, directories, filenames in os.walk(MYDRIVE):

    folder_path = Path(folder)

    relative = folder_path.relative_to(MYDRIVE)

    # Exclude completed synthetic forecasting trees.
    directories[:] = [
        name
        for name in directories
        if name not in skip_dirs
        and not name.startswith("Synthetic_NAF")
        and "test" not in name.lower()
    ]

    # Avoid excessive directory depth.
    if len(relative.parts) >= 9:
        directories[:] = []
        continue

    for filename in filenames:

        if filename.lower() not in expected_names:
            continue

        path = folder_path / filename

        # Never open these files here.
        found.append(path)

if found:

    for path in sorted(found):

        print(
            "FOUND:",
            path.relative_to(MYDRIVE),
            "|",
            round(
                path.stat().st_size / 1024**2,
                2
            ),
            "MB"
        )

else:

    print(
        "No canonical UNSW-NB15 "
        "training-set CSV found in MyDrive."
    )

# ------------------------------------------------------------
# D. SAFETY STATUS
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("CELL 4 COMPLETE")
print("=" * 85)

print("Raw feature metadata inspected: YES")
print("Raw dataset structure inspected: YES")
print("Raw datasets fully loaded: NO")
print("Official TEST accessed: NO")
print("Models trained: NO")
print("Files modified: NO")

print("=" * 85)


In [ ]:

# =============================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 5: CANONICAL UNSW TRAIN-ONLY DATASET RESTORATION
#
# Downloads ONLY the training partition.
# Validates pinned file SHA256, shape and class counts.
#
# NO TEST ACCESS
# NO MODEL TRAINING
# NO GOOGLE DRIVE MODIFICATION
# =============================================================

from pathlib import Path
import hashlib
import requests
import pandas as pd

print("=" * 80)
print("CELL 5 — RESTORE CANONICAL UNSW-NB15 TRAIN DATA")
print("=" * 80)

# Publicly hosted, revision-pinned TRAIN-only copy.
TRAIN_URL = (
    "https://huggingface.co/datasets/"
    "Mouwiya/UNSW-NB15/resolve/"
    "8ad6b19373f143086a66f52b0d6035a6959d83a9/"
    "UNSW_NB15_training-set.csv"
)

# Published checksum of this particular pinned file.
EXPECTED_SHA256 = (
    "bec7dd5ec88dc2a0ccc7a07879d338395ed7421750f675fd0339e07dfe0648fa"
)

TRAIN_PATH = Path(
    "/content/UNSW_NB15_training-set.csv"
)

TEMP_PATH = Path(
    "/content/UNSW_NB15_training-set.csv.partial"
)

def file_sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024), b""
        ):
            digest.update(chunk)
    return digest.hexdigest()

# -------------------------------------------------------------
# 1. Download only when the local TRAIN file is absent.
# -------------------------------------------------------------

if not TRAIN_PATH.is_file():

    print("\nDownloading TRAIN partition only...")

    digest = hashlib.sha256()

    try:
        with requests.get(
            TRAIN_URL,
            stream=True,
            timeout=(30, 180)
        ) as response:

            response.raise_for_status()

            with TEMP_PATH.open("wb") as handle:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        digest.update(chunk)
                        handle.write(chunk)

        actual_hash = digest.hexdigest()

        if actual_hash != EXPECTED_SHA256:
            raise RuntimeError(
                "TRAIN download SHA256 mismatch. "
                "File rejected."
            )

        TEMP_PATH.replace(TRAIN_PATH)

    finally:
        if TEMP_PATH.exists():
            TEMP_PATH.unlink()

else:
    print("\nExisting temporary TRAIN file found.")

# -------------------------------------------------------------
# 2. Check exact downloaded file integrity.
# -------------------------------------------------------------

actual_hash = file_sha256(TRAIN_PATH)

print("\nSHA256:", actual_hash)

if actual_hash != EXPECTED_SHA256:
    raise RuntimeError(
        "Local TRAIN file does not match pinned source."
    )

print("TRAIN file checksum: PASS")

# -------------------------------------------------------------
# 3. Read TRAIN ONLY.
# -------------------------------------------------------------

train_df = pd.read_csv(
    TRAIN_PATH,
    low_memory=False
)

print("\nTRAIN shape:", train_df.shape)

if train_df.shape != (175341, 45):
    raise RuntimeError(
        "Unexpected TRAIN shape. "
        f"Received {train_df.shape}."
    )

required_columns = {
    "id", "attack_cat", "label",
    "proto", "service", "state"
}

missing = required_columns - set(train_df.columns)

if missing:
    raise RuntimeError(
        f"Missing required columns: {missing}"
    )

print("TRAIN schema: PASS")

# -------------------------------------------------------------
# 4. Verify published TRAIN class counts.
# -------------------------------------------------------------

expected_counts = {
    "Normal": 56000,
    "Generic": 40000,
    "Exploits": 33393,
    "Fuzzers": 18184,
    "DoS": 12264,
    "Reconnaissance": 10491,
    "Analysis": 2000,
    "Backdoor": 1746,
    "Shellcode": 1133,
    "Worms": 130
}

classes = (
    train_df["attack_cat"]
    .astype("string")
    .str.strip()
)

observed_counts = classes.value_counts(
    dropna=False
).to_dict()

print("\nTRAIN CLASS DISTRIBUTION")
print("-" * 80)

for class_name, count in expected_counts.items():

    observed = observed_counts.get(
        class_name, 0
    )

    print(
        f"{class_name:<18} "
        f"expected={count:>6} "
        f"observed={observed:>6}"
    )

if observed_counts != expected_counts:
    raise RuntimeError(
        "TRAIN class distribution does not match "
        "the canonical training partition."
    )

print("\nClass distribution: PASS")

# -------------------------------------------------------------
# 5. Verify attack/normal label agreement.
# -------------------------------------------------------------

expected_binary = (
    classes != "Normal"
).astype(int)

actual_binary = pd.to_numeric(
    train_df["label"],
    errors="coerce"
)

label_mismatches = int(
    (expected_binary != actual_binary).sum()
)

print("Binary label mismatches:", label_mismatches)

if label_mismatches:
    raise RuntimeError(
        "TRAIN binary labels are inconsistent."
    )

print("\n" + "=" * 80)
print("CELL 5 — TRAIN DATA VERIFIED")
print("=" * 80)

print("Rows:", len(train_df))
print("Columns:", len(train_df.columns))
print("Attack classes including Normal:", len(expected_counts))
print("TRAIN checksum: VERIFIED")
print("Class counts: VERIFIED")
print("Binary labels: VERIFIED")

print("\nStored in temporary Colab storage:", TRAIN_PATH)
print("Official TEST accessed: NO")
print("Models trained: NO")
print("Google Drive files modified: NO")

print("=" * 80)


In [ ]:

# ================================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 6: LEAKAGE-AWARE TRAIN / VALIDATION SPLIT
#
# Uses ONLY the verified 175,341-row training partition.
# No official TEST access.
# No training or preprocessing fitting yet.
# ================================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold

print("=" * 85)
print("CELL 6 — TRAIN / VALIDATION SPLIT")
print("=" * 85)

if "train_df" not in globals():
    raise RuntimeError(
        "Run Cell 5 first to load the verified TRAIN dataset."
    )

df = train_df.copy()

TARGET = "attack_cat"
NON_FEATURE_COLUMNS = {
    "id", "attack_cat", "label"
}

feature_cols = [
    col for col in df.columns
    if col not in NON_FEATURE_COLUMNS
]

X = df[feature_cols].copy()
y = df[TARGET].astype(str).str.strip()

print("\nTotal rows:", len(df))
print("Feature count:", len(feature_cols))
print("Target classes:", y.nunique())

# -------------------------------------------------------------
# 1. Detect identical feature records.
# -------------------------------------------------------------

groups = pd.util.hash_pandas_object(
    X,
    index=False
)

duplicate_count = int(groups.duplicated().sum())

print("\nDuplicate feature rows:", duplicate_count)
print("Unique feature groups:", groups.nunique())

# Check whether identical feature records can carry
# different target labels.

label_count_per_group = (
    pd.DataFrame({
        "group": groups.to_numpy(),
        "label": y.to_numpy()
    })
    .groupby("group")["label"]
    .nunique()
)

conflicting_groups = int(
    (label_count_per_group > 1).sum()
)

print(
    "Groups with conflicting class labels:",
    conflicting_groups
)

# -------------------------------------------------------------
# 2. Group-aware stratified cross-validation split.
#
# All identical feature rows remain in the same partition.
# We select one validation fold closest to 20% and
# to the overall class proportions.
# -------------------------------------------------------------

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

overall_proportions = (
    y.value_counts(normalize=True)
)

candidates = []

for fold_id, (tr_idx, va_idx) in enumerate(
    splitter.split(X, y, groups)
):

    val_proportions = (
        y.iloc[va_idx]
        .value_counts(normalize=True)
        .reindex(
            overall_proportions.index,
            fill_value=0
        )
    )

    distribution_error = (
        val_proportions
        - overall_proportions
    ).abs().sum()

    size_error = abs(
        len(va_idx) / len(df) - 0.20
    )

    score = distribution_error + size_error

    candidates.append(
        (score, fold_id, tr_idx, va_idx)
    )

_, selected_fold, train_idx, val_idx = min(
    candidates,
    key=lambda item: item[0]
)

X_train = X.iloc[train_idx].copy()
X_val = X.iloc[val_idx].copy()

y_train = y.iloc[train_idx].copy()
y_val = y.iloc[val_idx].copy()

# -------------------------------------------------------------
# 3. Verify train/validation separation.
# -------------------------------------------------------------

train_groups = set(
    groups.iloc[train_idx].to_numpy()
)

val_groups = set(
    groups.iloc[val_idx].to_numpy()
)

overlapping_groups = len(
    train_groups.intersection(val_groups)
)

print("\nSelected fold:", selected_fold)
print("Train rows:", len(X_train))
print("Validation rows:", len(X_val))

print(
    "Validation percentage:",
    round(100 * len(X_val) / len(df), 2)
)

print(
    "Overlapping feature groups:",
    overlapping_groups
)

assert overlapping_groups == 0, (
    "Feature-group leakage detected."
)

assert (
    len(X_train) + len(X_val)
    == len(df)
)

assert set(y_train.unique()) == set(y.unique()), (
    "Training partition is missing a class."
)

assert set(y_val.unique()) == set(y.unique()), (
    "Validation partition is missing a class."
)

# -------------------------------------------------------------
# 4. Per-class distribution.
# -------------------------------------------------------------

distribution = pd.DataFrame({
    "Full TRAIN": y.value_counts(),
    "Development TRAIN": y_train.value_counts(),
    "Development VALIDATION": y_val.value_counts()
}).fillna(0).astype(int)

print("\nCLASS DISTRIBUTION")
print("-" * 85)

print(
    distribution
    .sort_index()
    .to_string()
)

# -------------------------------------------------------------
# 5. Summary.
# -------------------------------------------------------------

print("\n" + "=" * 85)
print("CELL 6 COMPLETE")
print("=" * 85)

print("Feature-group overlap: ZERO")
print("All 10 classes represented: YES")
print("Preprocessing fitted: NO")
print("Models trained: NO")
print("Official TEST accessed: NO")
print("Files modified: NO")

print("=" * 85)


In [ ]:

# ================================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 7: SINGLE XGBOOST MODEL TRAINING + VALIDATION
#
# One training run only.
# Uses verified development TRAIN/VALIDATION from Cell 6.
# No hyperparameter search.
# No official TEST access.
# No existing forecasting checkpoint modification.
# ================================================================

import time
import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support
)

print("=" * 85)
print("CELL 7 — SINGLE CURRENT ATTACK XGBOOST TRAINING")
print("=" * 85)

print("XGBoost version:", xgb.__version__)
print("Scikit-learn version:", sklearn.__version__)

required = [
    "X_train", "X_val",
    "y_train", "y_val"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run Cell 6 first."
        )

# ---------------------------------------------------------------
# 1. Preserve original feature schema.
# ---------------------------------------------------------------

CATEGORICAL_COLS = [
    c for c in ["proto", "service", "state"]
    if c in X_train.columns
]

NUMERICAL_COLS = [
    c for c in X_train.columns
    if c not in CATEGORICAL_COLS
]

FEATURE_COLUMNS = X_train.columns.tolist()

print("\nCategorical columns:", CATEGORICAL_COLS)
print("Numerical feature count:", len(NUMERICAL_COLS))
print("Total feature count:", len(FEATURE_COLUMNS))

def prepare_inputs(frame):
    result = frame[FEATURE_COLUMNS].copy()

    for col in CATEGORICAL_COLS:
        result[col] = (
            result[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    for col in NUMERICAL_COLS:
        result[col] = pd.to_numeric(
            result[col],
            errors="coerce"
        )
        result[col] = result[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

    return result

Xtr = prepare_inputs(X_train)
Xva = prepare_inputs(X_val)

# ---------------------------------------------------------------
# 2. Encode targets.
# ---------------------------------------------------------------

label_encoder = LabelEncoder()

ytr = label_encoder.fit_transform(y_train)

yva = label_encoder.transform(y_val)

CLASS_NAMES = label_encoder.classes_.tolist()
NORMAL_CLASS_ID = CLASS_NAMES.index("Normal")

print("\nClasses:", CLASS_NAMES)

# ---------------------------------------------------------------
# 3. TRAIN-only preprocessing.
# ---------------------------------------------------------------

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            NUMERICAL_COLS
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=True
            ),
            CATEGORICAL_COLS
        ),
    ],
    sparse_threshold=1.0
)

print("\nFitting preprocessing on TRAIN only...")

t0 = time.time()

Xtr_processed = preprocessor.fit_transform(Xtr)
Xva_processed = preprocessor.transform(Xva)

print(
    "Processed TRAIN shape:",
    Xtr_processed.shape
)
print(
    "Processed VALIDATION shape:",
    Xva_processed.shape
)

# ---------------------------------------------------------------
# 4. One multiclass XGBoost model.
# ---------------------------------------------------------------

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(CLASS_NAMES),

    n_estimators=180,
    max_depth=6,
    learning_rate=0.10,

    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,

    tree_method="hist",
    n_jobs=4,
    random_state=42,

    eval_metric="mlogloss"
)

print("\nTraining ONE XGBoost classifier...")

model.fit(
    Xtr_processed,
    ytr,
    verbose=False
)

training_seconds = time.time() - t0

print(
    "Training + preprocessing time:",
    round(training_seconds, 2),
    "seconds"
)

# ---------------------------------------------------------------
# 5. Multiclass validation.
# ---------------------------------------------------------------

proba = model.predict_proba(Xva_processed)
pred = np.argmax(proba, axis=1)

accuracy = accuracy_score(yva, pred)

macro_f1 = f1_score(
    yva, pred, average="macro",
    zero_division=0
)

balanced_acc = balanced_accuracy_score(
    yva, pred
)

weighted_f1 = f1_score(
    yva, pred, average="weighted",
    zero_division=0
)

print("\n" + "=" * 85)
print("MULTICLASS VALIDATION RESULTS")
print("=" * 85)

print(f"Accuracy          : {accuracy:.4f}")
print(f"Balanced accuracy : {balanced_acc:.4f}")
print(f"Macro F1          : {macro_f1:.4f}")
print(f"Weighted F1       : {weighted_f1:.4f}")

print("\nPER-CLASS REPORT")
print("-" * 85)

print(
    classification_report(
        yva,
        pred,
        labels=np.arange(len(CLASS_NAMES)),
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")
print("-" * 85)

cm = confusion_matrix(
    yva, pred,
    labels=np.arange(len(CLASS_NAMES))
)

print(
    pd.DataFrame(
        cm,
        index=CLASS_NAMES,
        columns=CLASS_NAMES
    ).to_string()
)

# ---------------------------------------------------------------
# 6. Derive current Attack / Normal probability.
#
# P(Attack) = 1 - P(Normal)
# This is a derived output, NOT a separately trained detector.
# ---------------------------------------------------------------

p_attack = 1.0 - proba[:, NORMAL_CLASS_ID]

true_binary = (yva != NORMAL_CLASS_ID).astype(int)

pred_binary = (p_attack >= 0.5).astype(int)

print("\n" + "=" * 85)
print("DERIVED CURRENT ATTACK / NORMAL RESULTS")
print("=" * 85)

print(
    classification_report(
        true_binary,
        pred_binary,
        target_names=["Normal", "Attack"],
        digits=4,
        zero_division=0
    )
)

print("Binary confusion matrix:")
print(
    confusion_matrix(
        true_binary,
        pred_binary
    )
)

# ---------------------------------------------------------------
# 7. Diagnose validation ambiguity.
#
# Identical feature vectors with conflicting labels
# are difficult or impossible to identify uniquely.
# We report them, but DO NOT silently remove them.
# ---------------------------------------------------------------

if (
    "groups" in globals()
    and "label_count_per_group" in globals()
    and "val_idx" in globals()
):

    val_hashes = groups.iloc[val_idx].to_numpy()

    ambiguous_mask = (
        label_count_per_group
        .reindex(val_hashes)
        .to_numpy() > 1
    )

    print("\n" + "=" * 85)
    print("VALIDATION LABEL AMBIGUITY")
    print("=" * 85)

    print(
        "Ambiguous validation records:",
        int(ambiguous_mask.sum())
    )

    print(
        "Unambiguous validation records:",
        int((~ambiguous_mask).sum())
    )

    for title, mask in [
        ("UNAMBIGUOUS", ~ambiguous_mask),
        ("AMBIGUOUS", ambiguous_mask)
    ]:

        if not mask.any():
            continue

        print(
            f"{title} accuracy:",
            round(
                accuracy_score(
                    yva[mask],
                    pred[mask]
                ),
                4
            )
        )

# ---------------------------------------------------------------
# 8. Keep trained objects in Colab memory.
# We'll save them permanently AFTER evaluating results.
# ---------------------------------------------------------------

CURRENT_ATTACK_MODEL = model
CURRENT_ATTACK_PREPROCESSOR = preprocessor
CURRENT_ATTACK_LABEL_ENCODER = label_encoder
CURRENT_ATTACK_FEATURE_COLUMNS = FEATURE_COLUMNS

print("\n" + "=" * 85)
print("CELL 7 COMPLETE")
print("=" * 85)

print("Models trained: 1")
print("Official TEST accessed: NO")
print("Forecasting models changed: NO")
print("Drive files modified: NO")
print("Checkpoint saved permanently: NOT YET")
print("=" * 85)


In [ ]:

# ================================================================
# NETGUARD CURRENT ATTACK DETECTION
# CELL 8 — SAVE TRAINED MODEL AND VERIFY RELOAD
#
# No retraining.
# No TEST access.
# Future forecasting artifacts untouched.
# ================================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import joblib
import numpy as np
import sklearn
import xgboost as xgb

print("=" * 80)
print("CELL 8 — SAVE CURRENT ATTACK CLASSIFIER")
print("=" * 80)

required = [
    "CURRENT_ATTACK_MODEL",
    "CURRENT_ATTACK_PREPROCESSOR",
    "CURRENT_ATTACK_LABEL_ENCODER",
    "CURRENT_ATTACK_FEATURE_COLUMNS",
    "Xva_processed",
    "proba",
    "accuracy",
    "macro_f1",
    "balanced_acc",
    "weighted_f1",
    "cm"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run Cell 7 first."
        )

# --------------------------------------------------------------
# 1. Create a NEW output directory.
# --------------------------------------------------------------

BASE_DIR = Path(
    "/content/drive/MyDrive/"
    "network-attack-forecasting/"
    "Current_Attack_Detection/artifacts"
)

RUN_ID = datetime.now(
    timezone.utc
).strftime("xgboost_%Y%m%d_%H%M%S_%f")

SAVE_DIR = BASE_DIR / RUN_ID

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=False
)

print("\nSaving to:")
print(SAVE_DIR)

# --------------------------------------------------------------
# 2. Save model and preprocessing.
# --------------------------------------------------------------

MODEL_FILE = SAVE_DIR / "current_attack_xgboost.ubj"
PREPROCESSOR_FILE = SAVE_DIR / "preprocessor.joblib"
SCHEMA_FILE = SAVE_DIR / "model_schema.json"
METRICS_FILE = SAVE_DIR / "validation_metrics.json"

CURRENT_ATTACK_MODEL.save_model(
    str(MODEL_FILE)
)

joblib.dump(
    CURRENT_ATTACK_PREPROCESSOR,
    PREPROCESSOR_FILE
)

class_names = [
    str(name)
    for name in
    CURRENT_ATTACK_LABEL_ENCODER.classes_
]

schema = {
    "task": "current_attack_multiclass",
    "dataset": "UNSW-NB15 canonical TRAIN partition",
    "training_partition": "development TRAIN only",
    "classes": class_names,
    "normal_class": "Normal",
    "feature_columns": list(
        CURRENT_ATTACK_FEATURE_COLUMNS
    ),
    "categorical_columns": list(
        CATEGORICAL_COLS
    ),
    "numerical_columns": list(
        NUMERICAL_COLS
    ),
    "input_preparation": {
        "categorical": (
            "pandas string, missing to "
            "__MISSING__, converted to str"
        ),
        "numeric": (
            "pandas.to_numeric(errors='coerce'), "
            "infinities converted to NaN"
        )
    },
    "binary_attack_probability": (
        "1 - P(Normal)"
    ),
    "binary_decision_threshold": 0.5,
    "python_libraries": {
        "xgboost": xgb.__version__,
        "scikit_learn": sklearn.__version__,
        "numpy": np.__version__
    }
}

with SCHEMA_FILE.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        schema,
        f,
        indent=2
    )

metrics = {
    "validation_rows": int(len(y_val)),
    "train_rows": int(len(y_train)),
    "accuracy": float(accuracy),
    "balanced_accuracy": float(balanced_acc),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "confusion_matrix": cm.tolist(),
    "class_order": class_names,
    "split": (
        "Cell 6 StratifiedGroupKFold "
        "selected development split"
    ),
    "official_test_used": False
}

with METRICS_FILE.open(
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        metrics,
        f,
        indent=2
    )

# --------------------------------------------------------------
# 3. Reload both artifacts and verify identical predictions.
# --------------------------------------------------------------

reloaded_model = xgb.XGBClassifier()
reloaded_model.load_model(
    str(MODEL_FILE)
)

reloaded_preprocessor = joblib.load(
    PREPROCESSOR_FILE
)

# Validate the saved preprocessing independently.
reloaded_Xva = reloaded_preprocessor.transform(
    Xva
)

reloaded_proba = reloaded_model.predict_proba(
    reloaded_Xva
)

np.testing.assert_allclose(
    reloaded_proba,
    proba,
    rtol=1e-6,
    atol=1e-7
)

print("\nMODEL RELOAD VERIFICATION: PASS")

# --------------------------------------------------------------
# 4. Save file hashes for integrity checking.
# --------------------------------------------------------------

def sha256_file(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

hashes = {}

for path in [
    MODEL_FILE,
    PREPROCESSOR_FILE,
    SCHEMA_FILE,
    METRICS_FILE
]:
    hashes[path.name] = sha256_file(path)

with (
    SAVE_DIR / "artifact_hashes.json"
).open("w", encoding="utf-8") as f:
    json.dump(hashes, f, indent=2)

# --------------------------------------------------------------
# 5. Final summary.
# --------------------------------------------------------------

print("\n" + "=" * 80)
print("CELL 8 COMPLETE — MODEL SAVED")
print("=" * 80)

for path in sorted(SAVE_DIR.iterdir()):
    print(
        path.name,
        "|",
        round(path.stat().st_size / 1024, 2),
        "KB"
    )

print("\nSaved directory:", SAVE_DIR)
print("Models retrained: 0")
print("Official TEST accessed: NO")
print("Future forecasting models modified: NO")
print("Reload prediction agreement: PASS")
print("=" * 80)


In [ ]:

# ================================================================
# NETGUARD CURRENT ATTACK DETECTION
# CELL 9 — SAVED MODEL INFERENCE PIPELINE
#
# No training
# No official TEST access
# No artifact modification
# ================================================================

import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from pathlib import Path

print("=" * 80)
print("CELL 9 — CURRENT ATTACK INFERENCE")
print("=" * 80)

ARTIFACT_DIR = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Current_Attack_Detection/artifacts/"
    "xgboost_20261027_023158_885060"
)

with (ARTIFACT_DIR / "model_schema.json").open(
    "r", encoding="utf-8"
) as f:
    schema = json.load(f)

classifier = xgb.XGBClassifier()

classifier.load_model(
    str(ARTIFACT_DIR / "current_attack_xgboost.ubj")
)

preprocessor_loaded = joblib.load(
    ARTIFACT_DIR / "preprocessor.joblib"
)

FEATURES = schema["feature_columns"]
CATEGORIES = schema["categorical_columns"]
NUMERICS = schema["numerical_columns"]
CLASSES = schema["classes"]

NORMAL_IDX = CLASSES.index("Normal")

def predict_current_attack(records, threshold=0.5):

    if isinstance(records, dict):
        records = pd.DataFrame([records])

    if not isinstance(records, pd.DataFrame):
        raise TypeError(
            "Expected a DataFrame or one record dictionary."
        )

    missing = [
        c for c in FEATURES
        if c not in records.columns
    ]

    if missing:
        raise ValueError(
            f"Missing model features: {missing}"
        )

    frame = records[FEATURES].copy()

    # Match the original training-time input conversion.

    for col in CATEGORIES:
        frame[col] = (
            frame[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    for col in NUMERICS:
        frame[col] = pd.to_numeric(
            frame[col],
            errors="coerce"
        )

        frame[col] = frame[col].replace(
            [np.inf, -np.inf],
            np.nan
        )

    X_processed = preprocessor_loaded.transform(frame)

    probabilities = classifier.predict_proba(
        X_processed
    )

    results = []

    for row in probabilities:

        p_normal = float(row[NORMAL_IDX])
        p_attack = float(1.0 - p_normal)

        attack_indices = [
            i for i, name in enumerate(CLASSES)
            if name != "Normal"
        ]

        best_attack_idx = max(
            attack_indices,
            key=lambda i: row[i]
        )

        attack_detected = p_attack >= threshold

        results.append({
            "current_attack_detected": bool(
                attack_detected
            ),
            "attack_probability": round(
                p_attack, 6
            ),
            "current_attack_type": (
                CLASSES[best_attack_idx]
                if attack_detected
                else "Normal"
            ),
            "attack_type_probability": (
                round(float(row[best_attack_idx]), 6)
                if attack_detected
                else None
            ),
            "attack_probabilities": {
                name: round(float(row[i]), 6)
                for i, name in enumerate(CLASSES)
            },
            "model": "XGBoost_Current_Attack_V1",
            "threshold": float(threshold)
        })

    return results


# ------------------------------------------------
# TEST ON DEVELOPMENT VALIDATION DATA ONLY
# ------------------------------------------------

if "X_val" not in globals() or "y_val" not in globals():
    raise RuntimeError(
        "Run Cell 6 to restore validation data, "
        "or supply new 42-feature network records."
    )

sample = X_val.head(5)
actual = y_val.head(5).tolist()

predictions = predict_current_attack(sample)

for i, (truth, prediction) in enumerate(
    zip(actual, predictions), start=1
):

    print(f"\nRECORD {i}")
    print("-" * 50)

    print("Actual class:", truth)

    print(
        "Attack detected:",
        prediction["current_attack_detected"]
    )

    print(
        "Attack probability:",
        f'{prediction["attack_probability"]:.2%}'
    )

    print(
        "Predicted class:",
        prediction["current_attack_type"]
    )

    print(
        "Attack-type probability:",
        prediction["attack_type_probability"]
    )

    print(
        "All ten probabilities:",
        prediction["attack_probabilities"]
    )

print("\n" + "=" * 80)
print("CELL 9 COMPLETE")
print("Saved model loaded: YES")
print("New training: NO")
print("Official TEST accessed: NO")
print("Forecasting models modified: NO")
print("=" * 80)


In [ ]:

# ============================================================
# NETGUARD CURRENT ATTACK DETECTION
# CELL 10 — AP, PR-AUC AND ERROR ANALYSIS
#
# Uses existing development-validation predictions.
# No retraining, no TEST access, no file writes.
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    f1_score
)

print("=" * 80)
print("CELL 10 — CURRENT ATTACK MODEL QUALITY")
print("=" * 80)

# Reuse Cell 7 validation results.
p = np.asarray(proba)
y = np.asarray(yva)

normal_idx = CLASS_NAMES.index("Normal")
attack_truth = (y != normal_idx).astype(int)
attack_scores = 1.0 - p[:, normal_idx]

print("\nA. BINARY ATTACK DETECTION")
print("-" * 80)

attack_ap = average_precision_score(
    attack_truth, attack_scores
)

attack_auc = roc_auc_score(
    attack_truth, attack_scores
)

attack_prevalence = attack_truth.mean()

print(f"Attack prevalence : {attack_prevalence:.4f}")
print(f"Attack AP         : {attack_ap:.4f}")
print(f"ROC-AUC           : {attack_auc:.4f}")
print(
    f"Brier score       : "
    f"{brier_score_loss(attack_truth, attack_scores):.4f}"
)

print("\nB. THRESHOLD DIAGNOSTICS")
print("-" * 80)

rows = []

for threshold in [0.50, 0.70, 0.80, 0.90, 0.95]:

    pred_attack = (
        attack_scores >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        attack_truth,
        pred_attack,
        labels=[0, 1]
    ).ravel()

    precision = tp / (tp + fp) if tp + fp else 0
    recall = tp / (tp + fn) if tp + fn else 0
    fpr = fp / (fp + tn) if fp + tn else 0

    rows.append({
        "Threshold": threshold,
        "Precision": round(precision, 4),
        "Attack recall": round(recall, 4),
        "False-positive rate": round(fpr, 4),
        "Binary F1": round(
            f1_score(attack_truth, pred_attack),
            4
        ),
        "FP": int(fp),
        "FN": int(fn)
    })

threshold_table = pd.DataFrame(rows)
print(threshold_table.to_string(index=False))

print("\nC. PER-CLASS AVERAGE PRECISION")
print("-" * 80)

class_rows = []

for i, name in enumerate(CLASS_NAMES):

    class_truth = (y == i).astype(int)

    class_ap = average_precision_score(
        class_truth,
        p[:, i]
    )

    class_rows.append({
        "Class": name,
        "Support": int(class_truth.sum()),
        "Prevalence": round(
            float(class_truth.mean()), 5
        ),
        "AP": round(float(class_ap), 4)
    })

class_table = pd.DataFrame(class_rows)

print(
    class_table.sort_values("AP")
    .to_string(index=False)
)

macro_ap = np.mean([
    average_precision_score(
        (y == i).astype(int),
        p[:, i]
    )
    for i in range(len(CLASS_NAMES))
])

print(f"\nMacro one-vs-rest AP: {macro_ap:.4f}")

print("\nD. BINARY PROBABILITY CALIBRATION")
print("-" * 80)

calibration_rows = []
edges = np.linspace(0, 1, 11)
ece = 0.0

for i in range(10):

    low, high = edges[i], edges[i + 1]

    mask = (
        (attack_scores >= low)
        & (
            (attack_scores <= high)
            if i == 9
            else (attack_scores < high)
        )
    )

    if not mask.any():
        continue

    mean_pred = float(attack_scores[mask].mean())
    actual_rate = float(attack_truth[mask].mean())
    count = int(mask.sum())

    ece += (
        count / len(attack_truth)
    ) * abs(mean_pred - actual_rate)

    calibration_rows.append({
        "Score band": f"{low:.1f}-{high:.1f}",
        "Records": count,
        "Mean predicted": round(mean_pred, 4),
        "Actual attack rate": round(actual_rate, 4)
    })

print(
    pd.DataFrame(calibration_rows)
    .to_string(index=False)
)

print(f"\n10-bin ECE: {ece:.4f}")

print("\n" + "=" * 80)
print("CELL 10 COMPLETE")
print("=" * 80)
print("Models retrained: NO")
print("Official TEST accessed: NO")
print("Existing artifacts modified: NO")


In [ ]:

# ============================================================
# CELL 11 — TARGETED CLASS DECISION OPTIMIZATION
#
# NO RETRAINING
# NO TEST ACCESS
# NO MODEL OR DRIVE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)

print("=" * 80)
print("CELL 11 — IMPROVE DoS / BACKDOOR / ANALYSIS")
print("=" * 80)

required = [
    "proba", "yva", "CLASS_NAMES",
    "groups", "val_idx"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Restore earlier cells first."
        )

P = np.asarray(proba)
Y = np.asarray(yva)
NAMES = list(CLASS_NAMES)

assert P.shape == (len(Y), len(NAMES))

# ------------------------------------------------------------
# 1. Separate tuning and evaluation groups.
# ------------------------------------------------------------

validation_groups = (
    groups.iloc[val_idx].to_numpy()
)

splitter = StratifiedGroupKFold(
    n_splits=2,
    shuffle=True,
    random_state=2026
)

tune_idx, check_idx = next(
    splitter.split(
        np.zeros((len(Y), 1)),
        Y,
        validation_groups
    )
)

assert not (
    set(validation_groups[tune_idx])
    & set(validation_groups[check_idx])
)

P_tune = P[tune_idx]
Y_tune = Y[tune_idx]

P_check = P[check_idx]
Y_check = Y[check_idx]

print("\nDecision tuning rows:", len(Y_tune))
print("Held-out check rows:", len(Y_check))
print("Overlapping feature groups: 0")

# ------------------------------------------------------------
# 2. Optimize weights ONLY on tuning portion.
# ------------------------------------------------------------

TARGET_CLASSES = [
    "DoS", "Backdoor", "Analysis"
]

CANDIDATE_WEIGHTS = [
    1.0, 1.25, 1.5, 2.0,
    2.5, 3.0, 4.0, 6.0, 8.0
]

weights = np.ones(
    len(NAMES),
    dtype=float
)

def weighted_predict(probabilities, class_weights):
    return np.argmax(
        probabilities * class_weights,
        axis=1
    )

baseline_tune_predictions = np.argmax(
    P_tune,
    axis=1
)

baseline_tune_accuracy = accuracy_score(
    Y_tune,
    baseline_tune_predictions
)

minimum_accuracy = (
    baseline_tune_accuracy - 0.02
)

def score_weights(candidate_weights):

    predictions = weighted_predict(
        P_tune,
        candidate_weights
    )

    acc = accuracy_score(
        Y_tune,
        predictions
    )

    macro = f1_score(
        Y_tune,
        predictions,
        average="macro",
        labels=np.arange(len(NAMES)),
        zero_division=0
    )

    if acc < minimum_accuracy:
        return -1

    return macro

best_score = score_weights(weights)

print("\nBaseline tuning macro F1:",
      round(best_score, 4))

print("Minimum allowed tuning accuracy:",
      round(minimum_accuracy, 4))

for iteration in range(3):

    improved = False

    for class_name in TARGET_CLASSES:

        class_id = NAMES.index(class_name)

        for weight in CANDIDATE_WEIGHTS:

            candidate = weights.copy()
            candidate[class_id] = weight

            candidate_score = score_weights(
                candidate
            )

            if candidate_score > best_score + 1e-6:

                best_score = candidate_score
                weights = candidate
                improved = True

    if not improved:
        break

print("\nSELECTED DECISION WEIGHTS")

for name in TARGET_CLASSES:
    print(
        f"{name:<12}: "
        f"{weights[NAMES.index(name)]:.2f}"
    )

# ------------------------------------------------------------
# 3. Evaluate on untouched check portion.
# ------------------------------------------------------------

baseline_pred = np.argmax(
    P_check,
    axis=1
)

adjusted_pred = weighted_predict(
    P_check,
    weights
)

def summary_metrics(y_true, y_pred):

    return {
        "Accuracy": accuracy_score(
            y_true, y_pred
        ),
        "Balanced accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "Macro F1": f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    }

baseline_metrics = summary_metrics(
    Y_check,
    baseline_pred
)

adjusted_metrics = summary_metrics(
    Y_check,
    adjusted_pred
)

print("\n" + "=" * 80)
print("HELD-OUT CHECK — OVERALL METRICS")
print("=" * 80)

for metric in baseline_metrics:

    old = baseline_metrics[metric]
    new = adjusted_metrics[metric]

    print(
        f"{metric:<22}"
        f"Before: {old:.4f}  "
        f"After: {new:.4f}  "
        f"Change: {new-old:+.4f}"
    )

# ------------------------------------------------------------
# 4. Per-class precision, recall and F1.
# ------------------------------------------------------------

def class_report(predictions):

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            Y_check,
            predictions,
            labels=np.arange(len(NAMES)),
            zero_division=0
        )
    )

    return pd.DataFrame({
        "Class": NAMES,
        "Support": support,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

before = class_report(baseline_pred)
after = class_report(adjusted_pred)

comparison = pd.DataFrame({
    "Class": NAMES,
    "Support": before["Support"],
    "Recall before": before["Recall"],
    "Recall after": after["Recall"],
    "Precision before": before["Precision"],
    "Precision after": after["Precision"],
    "F1 before": before["F1"],
    "F1 after": after["F1"]
})

print("\n" + "=" * 80)
print("HELD-OUT CHECK — PER-CLASS COMPARISON")
print("=" * 80)

print(
    comparison.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 5. Check whether binary detection gets worse.
# ------------------------------------------------------------

normal_id = NAMES.index("Normal")

binary_truth = (
    Y_check != normal_id
).astype(int)

print("\n" + "=" * 80)
print("BINARY DETECTION IMPACT")
print("=" * 80)

for title, predictions in [
    ("Original", baseline_pred),
    ("Adjusted", adjusted_pred)
]:

    binary_predictions = (
        predictions != normal_id
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        binary_truth,
        binary_predictions,
        labels=[0, 1]
    ).ravel()

    print(
        f"{title:<10} | "
        f"False positives: {fp:>5} | "
        f"Missed attacks: {fn:>5} | "
        f"Attack recall: {tp/(tp+fn):.4f}"
    )

# ------------------------------------------------------------
# 6. Keep candidate weights in memory only.
# ------------------------------------------------------------

CURRENT_ATTACK_CANDIDATE_WEIGHTS = {
    name: float(weights[i])
    for i, name in enumerate(NAMES)
}

print("\n" + "=" * 80)
print("CELL 11 COMPLETE")
print("=" * 80)
print("New models trained: 0")
print("Original model modified: NO")
print("Official TEST accessed: NO")
print("Candidate decision weights saved to Drive: NO")
print("=" * 80)


In [ ]:

# ============================================================
# CELL 12 — VERIFY TWO-STAGE CURRENT ATTACK CLASSIFICATION
#
# Stage 1: Original binary attack probability threshold
# Stage 2: Weighted attack-family choice, only if attack
#
# NO RETRAINING
# NO NEW WEIGHT TUNING
# NO OFFICIAL TEST ACCESS
# NO FILE MODIFICATIONS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    precision_recall_fscore_support
)

print("=" * 80)
print("CELL 12 — TWO-STAGE DECISION VALIDATION")
print("=" * 80)

required = [
    "P_check", "Y_check", "NAMES", "weights"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run Cell 11 first."
        )

P = np.asarray(P_check)
Y = np.asarray(Y_check)
NAMES = list(NAMES)
W = np.asarray(weights, dtype=float)

assert P.shape == (len(Y), len(NAMES))
assert W.shape == (len(NAMES),)
assert np.all(W > 0)
assert "Normal" in NAMES

normal_id = NAMES.index("Normal")
attack_ids = np.array([
    i for i, name in enumerate(NAMES)
    if name != "Normal"
])

THRESHOLD = 0.50

# ------------------------------------------------------------
# 1. Separate binary detection from attack-family selection.
# ------------------------------------------------------------

p_attack = 1.0 - P[:, normal_id]

detected = p_attack >= THRESHOLD

def two_stage_predict(class_weights):

    # Weights select the attack family ONLY.
    # They do not alter P(Attack).
    attack_scores = (
        P[:, attack_ids]
        * class_weights[attack_ids]
    )

    selected_attack = attack_ids[
        np.argmax(attack_scores, axis=1)
    ]

    return np.where(
        detected,
        selected_attack,
        normal_id
    )

original_weights = np.ones(len(NAMES))

pred_original = two_stage_predict(
    original_weights
)

pred_adjusted = two_stage_predict(
    W
)

# ------------------------------------------------------------
# 2. Full ten-class comparison.
# ------------------------------------------------------------

def overall_metrics(pred):

    return {
        "Accuracy": accuracy_score(Y, pred),
        "Balanced accuracy": balanced_accuracy_score(
            Y, pred
        ),
        "Macro F1": f1_score(
            Y, pred,
            labels=np.arange(len(NAMES)),
            average="macro",
            zero_division=0
        )
    }

before = overall_metrics(pred_original)
after = overall_metrics(pred_adjusted)

print("\nA. OVERALL CLASSIFICATION")
print("-" * 80)

for metric in before:

    print(
        f"{metric:<22} "
        f"Before={before[metric]:.4f}  "
        f"After={after[metric]:.4f}  "
        f"Delta={after[metric]-before[metric]:+.4f}"
    )

# ------------------------------------------------------------
# 3. Detailed per-class performance.
# ------------------------------------------------------------

def per_class(pred):

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            Y,
            pred,
            labels=np.arange(len(NAMES)),
            zero_division=0
        )
    )

    return pd.DataFrame({
        "Class": NAMES,
        "Support": support,
        "Precision": precision,
        "Recall": recall,
        "F1": f1
    })

old = per_class(pred_original)
new = per_class(pred_adjusted)

comparison = pd.DataFrame({
    "Class": NAMES,
    "Support": old["Support"],
    "Recall before": old["Recall"],
    "Recall after": new["Recall"],
    "Precision before": old["Precision"],
    "Precision after": new["Precision"],
    "F1 before": old["F1"],
    "F1 after": new["F1"]
})

print("\nB. PER-CLASS COMPARISON")
print("-" * 80)

print(comparison.round(4).to_string(index=False))

# ------------------------------------------------------------
# 4. Confirm binary detector stays unchanged.
# ------------------------------------------------------------

binary_truth = (
    Y != normal_id
).astype(int)

print("\nC. BINARY DETECTION CONSISTENCY")
print("-" * 80)

for title, pred in [
    ("Unweighted", pred_original),
    ("Weighted", pred_adjusted)
]:

    binary_pred = (
        pred != normal_id
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        binary_truth,
        binary_pred,
        labels=[0, 1]
    ).ravel()

    print(
        f"{title:<12} "
        f"TP={tp:>6} FP={fp:>5} "
        f"FN={fn:>5} TN={tn:>6}"
    )

# The key safety invariant:
assert np.array_equal(
    pred_original != normal_id,
    pred_adjusted != normal_id
), "ERROR: Attack detection changed after family weighting."

print("\nBinary detector unchanged: PASS")

# ------------------------------------------------------------
# 5. Preserve candidate decision configuration in memory.
# ------------------------------------------------------------

TWO_STAGE_CANDIDATE = {
    "binary_threshold": THRESHOLD,
    "class_decision_weights": {
        name: float(W[i])
        for i, name in enumerate(NAMES)
    },
    "classification_rule": (
        "First detect attack using 1-P(Normal); "
        "if attack, choose argmax of weighted scores "
        "across the nine attack classes."
    )
}

print("\nD. CANDIDATE CONFIGURATION")
print("-" * 80)
print(TWO_STAGE_CANDIDATE)

print("\n" + "=" * 80)
print("CELL 12 COMPLETE")
print("=" * 80)
print("New models trained: 0")
print("Binary detection changed: NO")
print("Official TEST accessed: NO")
print("Saved checkpoint modified: NO")
print("=" * 80)


In [ ]:

# ============================================================
# CELL 13 — SAVE TWO-STAGE V1.1 DECISION CONFIGURATION
#
# NO RETRAINING
# NO OFFICIAL TEST ACCESS
# DOES NOT OVERWRITE XGBOOST V1
# DOES NOT MODIFY FORECASTING MODELS
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix
)

print("=" * 80)
print("CELL 13 — SAVE CURRENT ATTACK V1.1")
print("=" * 80)

required = [
    "TWO_STAGE_CANDIDATE",
    "P_check",
    "Y_check",
    "NAMES",
    "pred_original",
    "pred_adjusted"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Run Cell 12 first."
        )

MODEL_V1_DIR = Path(
    "/content/drive/MyDrive/network-attack-forecasting/"
    "Current_Attack_Detection/artifacts/"
    "xgboost_20261027_023158_885060"
)

assert (
    MODEL_V1_DIR / "current_attack_xgboost.ubj"
).is_file(), "Original V1 checkpoint missing."

assert (
    MODEL_V1_DIR / "preprocessor.joblib"
).is_file(), "Original V1 preprocessor missing."

CONFIG_ROOT = (
    MODEL_V1_DIR.parent.parent
    / "decision_configs"
)

RUN_ID = datetime.now(
    timezone.utc
).strftime("two_stage_v1_1_%Y%m%d_%H%M%S_%f")

CONFIG_DIR = CONFIG_ROOT / RUN_ID

CONFIG_DIR.mkdir(
    parents=True,
    exist_ok=False
)

# ------------------------------------------------------------
# 1. Save decision configuration.
# ------------------------------------------------------------

config = {
    "name": "NetGuard_Current_Attack_V1.1",
    "status": "experimental",
    "base_model_directory": str(MODEL_V1_DIR),
    "binary_threshold": float(
        TWO_STAGE_CANDIDATE["binary_threshold"]
    ),
    "class_decision_weights": dict(
        TWO_STAGE_CANDIDATE[
            "class_decision_weights"
        ]
    ),
    "class_order": list(NAMES),
    "normal_class": "Normal",
    "decision_rule": (
        "If 1-P(Normal) is below the binary threshold, "
        "output Normal. Otherwise, choose the attack "
        "class with the largest class probability "
        "multiplied by its decision weight."
    ),
    "probabilities_modified": False,
    "weight_selection": (
        "Tuned on group-separated development-validation "
        "subset; evaluated on a second subset."
    ),
    "official_test_evaluated": False
}

config_file = CONFIG_DIR / "decision_config.json"

with config_file.open(
    "x",
    encoding="utf-8"
) as f:
    json.dump(config, f, indent=2)

# ------------------------------------------------------------
# 2. Save evaluation summary.
# ------------------------------------------------------------

def calculate_metrics(predictions):

    return {
        "accuracy": float(
            accuracy_score(Y_check, predictions)
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                Y_check, predictions
            )
        ),
        "macro_f1": float(
            f1_score(
                Y_check,
                predictions,
                labels=np.arange(len(NAMES)),
                average="macro",
                zero_division=0
            )
        )
    }

def per_class_metrics(predictions):

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            Y_check,
            predictions,
            labels=np.arange(len(NAMES)),
            zero_division=0
        )
    )

    return {
        name: {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(support[i])
        }
        for i, name in enumerate(NAMES)
    }

evaluation = {
    "evaluation_type": "exploratory held-out check",
    "check_rows": int(len(Y_check)),
    "original_two_stage": {
        "overall": calculate_metrics(pred_original),
        "per_class": per_class_metrics(pred_original)
    },
    "weighted_two_stage": {
        "overall": calculate_metrics(pred_adjusted),
        "per_class": per_class_metrics(pred_adjusted)
    },
    "official_test_used": False
}

evaluation_file = (
    CONFIG_DIR / "evaluation.json"
)

with evaluation_file.open(
    "x",
    encoding="utf-8"
) as f:
    json.dump(evaluation, f, indent=2)

# ------------------------------------------------------------
# 3. Verify saved configuration.
# ------------------------------------------------------------

with config_file.open(
    "r",
    encoding="utf-8"
) as f:
    restored = json.load(f)

assert restored["class_order"] == list(NAMES)

restored_weights = np.array([
    restored["class_decision_weights"][name]
    for name in NAMES
], dtype=float)

assert np.allclose(
    restored_weights,
    np.asarray(weights, dtype=float)
)

print("\nConfiguration reload: PASS")
print("V1 model checkpoint: UNCHANGED")

print("\nSaved files:")
for path in sorted(CONFIG_DIR.iterdir()):
    print(" -", path.name)

print("\nV1.1 macro F1:",
      round(
          evaluation["weighted_two_stage"]["overall"]["macro_f1"],
          4
      ))

print("\n" + "=" * 80)
print("CELL 13 COMPLETE")
print("=" * 80)
print("Saved directory:", CONFIG_DIR)
print("Models trained: 0")
print("Original model overwritten: NO")
print("Official TEST accessed: NO")
print("Forecasting models modified: NO")
print("=" * 80)


In [ ]:

# ============================================================
# CELL 14 — TARGETED DoS vs EXPLOITS SPECIALIST
#
# ONE small specialist model.
# TRAIN partition only for fitting and threshold selection.
# Cell 11 held-out check subset for exploratory evaluation.
#
# NO official TEST access.
# NO Drive writes.
# NO changes to existing models.
# ============================================================

import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    precision_recall_fscore_support,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 85)
print("CELL 14 — DoS / EXPLOITS SPECIALIST")
print("=" * 85)

required = [
    "Xtr_processed", "Xva_processed",
    "ytr", "train_idx", "groups",
    "check_idx", "Y_check",
    "NAMES", "pred_adjusted"
]

for item in required:
    if item not in globals():
        raise RuntimeError(
            f"Missing {item}. Restore the earlier cells."
        )

names = list(NAMES)

dos_id = names.index("DoS")
exp_id = names.index("Exploits")
normal_id = names.index("Normal")

y_train_encoded = np.asarray(ytr)
y_check = np.asarray(Y_check)
baseline = np.asarray(pred_adjusted)

# ------------------------------------------------------------
# 1. Use only DoS and Exploits development TRAIN examples.
# ------------------------------------------------------------

specialist_mask = np.isin(
    y_train_encoded,
    [dos_id, exp_id]
)

specialist_indices = np.flatnonzero(
    specialist_mask
)

X_specialist = Xtr_processed[
    specialist_indices
]

y_specialist = (
    y_train_encoded[specialist_indices] == dos_id
).astype(int)

training_groups = (
    groups.iloc[train_idx]
    .to_numpy()[specialist_indices]
)

print("\nSPECIALIST TRAINING DATA")
print("-" * 85)

print("Total examples:", len(y_specialist))
print("DoS:", int(y_specialist.sum()))
print(
    "Exploits:",
    int((y_specialist == 0).sum())
)

# ------------------------------------------------------------
# 2. Internal group-separated split.
#    This split uses development TRAIN, not TEST.
# ------------------------------------------------------------

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=707
)

inner_train, inner_val = next(
    splitter.split(
        X_specialist,
        y_specialist,
        training_groups
    )
)

assert not (
    set(training_groups[inner_train])
    & set(training_groups[inner_val])
)

print("\nInternal training:", len(inner_train))
print("Internal validation:", len(inner_val))
print("Overlapping feature groups: 0")

# ------------------------------------------------------------
# 3. Train ONE specialist XGBoost.
# ------------------------------------------------------------

dos_train_count = int(
    y_specialist[inner_train].sum()
)

exp_train_count = int(
    (y_specialist[inner_train] == 0).sum()
)

class_ratio = (
    exp_train_count / max(1, dos_train_count)
)

specialist = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=220,
    max_depth=5,
    learning_rate=0.07,
    min_child_weight=3,
    subsample=0.85,
    colsample_bytree=0.85,
    tree_method="hist",
    scale_pos_weight=class_ratio,
    eval_metric="logloss",
    n_jobs=4,
    random_state=707
)

print("\nTraining ONE DoS specialist...")

specialist.fit(
    X_specialist[inner_train],
    y_specialist[inner_train],
    verbose=False
)

# ------------------------------------------------------------
# 4. Select thresholds using internal validation ONLY.
# ------------------------------------------------------------

inner_scores = specialist.predict_proba(
    X_specialist[inner_val]
)[:, 1]

inner_truth = y_specialist[inner_val]

precision, recall, thresholds = (
    precision_recall_curve(
        inner_truth,
        inner_scores
    )
)

# Exclude final PR endpoint, which has no threshold.
prec = precision[:-1]
rec = recall[:-1]

f1_values = (
    2 * prec * rec
    / np.maximum(prec + rec, 1e-12)
)

best_f1_index = int(
    np.argmax(f1_values)
)

f1_threshold = float(
    thresholds[best_f1_index]
)

# Find highest-precision threshold reaching >=90% recall
# on specialist's internal DoS-vs-Exploits validation.

eligible = np.flatnonzero(rec >= 0.90)

if len(eligible):

    target_index = max(
        eligible,
        key=lambda i: (
            float(prec[i]),
            float(thresholds[i])
        )
    )

    recall90_threshold = float(
        thresholds[target_index]
    )

else:

    recall90_threshold = None

print("\nINTERNAL VALIDATION")
print("-" * 85)

print(
    "DoS vs Exploits AP:",
    round(
        average_precision_score(
            inner_truth,
            inner_scores
        ),
        4
    )
)

print(
    "Best-F1 threshold:",
    round(f1_threshold, 6)
)

if recall90_threshold is not None:

    print(
        "90%-recall candidate threshold:",
        round(recall90_threshold, 6)
    )

    print(
        "Internal precision at candidate:",
        round(float(prec[target_index]), 4)
    )

    print(
        "Internal recall at candidate:",
        round(float(rec[target_index]), 4)
    )

# ------------------------------------------------------------
# 5. Route only DoS / Exploits decisions.
#    Keep original Attack/Normal decision unchanged.
# ------------------------------------------------------------

X_check_processed = Xva_processed[
    check_idx
]

route_mask = np.isin(
    baseline,
    [dos_id, exp_id]
)

print("\nROUTING COVERAGE")
print("-" * 85)

print(
    "Records sent to specialist:",
    int(route_mask.sum())
)

actual_dos_mask = (
    y_check == dos_id
)

print(
    "Actual DoS routed:",
    int((actual_dos_mask & route_mask).sum()),
    "of",
    int(actual_dos_mask.sum())
)

routed_scores = specialist.predict_proba(
    X_check_processed[route_mask]
)[:, 1]

def evaluate_candidate(title, cutoff):

    predictions = baseline.copy()

    predictions[route_mask] = np.where(
        routed_scores >= cutoff,
        dos_id,
        exp_id
    )

    # Binary attack detection must never change.
    assert np.array_equal(
        predictions != normal_id,
        baseline != normal_id
    )

    pr, re, f1, support = (
        precision_recall_fscore_support(
            y_check,
            predictions,
            labels=np.arange(len(names)),
            zero_division=0
        )
    )

    print("\n" + title)
    print("-" * 85)

    print(
        "Overall accuracy:",
        round(
            accuracy_score(y_check, predictions),
            4
        )
    )

    print(
        "Macro F1:",
        round(
            f1_score(
                y_check,
                predictions,
                average="macro",
                zero_division=0
            ),
            4
        )
    )

    print(
        "Balanced accuracy:",
        round(
            balanced_accuracy_score(
                y_check,
                predictions
            ),
            4
        )
    )

    for class_name in [
        "DoS", "Exploits",
        "Backdoor", "Analysis"
    ]:

        i = names.index(class_name)

        print(
            f"{class_name:<12} "
            f"Precision={pr[i]:.4f} "
            f"Recall={re[i]:.4f} "
            f"F1={f1[i]:.4f}"
        )

    return predictions

# ------------------------------------------------------------
# 6. Baseline and specialist comparisons.
# ------------------------------------------------------------

baseline_pr, baseline_re, baseline_f1, _ = (
    precision_recall_fscore_support(
        y_check,
        baseline,
        labels=np.arange(len(names)),
        zero_division=0
    )
)

print("\n" + "=" * 85)
print("EXISTING TWO-STAGE V1.1 BASELINE")
print("=" * 85)

print(
    "Overall accuracy:",
    round(accuracy_score(y_check, baseline), 4)
)

print(
    "Macro F1:",
    round(
        f1_score(
            y_check,
            baseline,
            average="macro",
            zero_division=0
        ),
        4
    )
)

print(
    "DoS precision:",
    round(float(baseline_pr[dos_id]), 4)
)

print(
    "DoS recall:",
    round(float(baseline_re[dos_id]), 4)
)

print(
    "DoS F1:",
    round(float(baseline_f1[dos_id]), 4)
)

specialist_f1_predictions = evaluate_candidate(
    "SPECIALIST — F1-OPTIMIZED THRESHOLD",
    f1_threshold
)

if recall90_threshold is not None:

    specialist_recall90_predictions = (
        evaluate_candidate(
            "SPECIALIST — 90% INTERNAL RECALL TARGET",
            recall90_threshold
        )
    )

else:

    specialist_recall90_predictions = None

# ------------------------------------------------------------
# 7. Keep everything in memory.
#    Do NOT replace the existing XGBoost or V1.1 config.
# ------------------------------------------------------------

DOS_SPECIALIST_MODEL = specialist

DOS_SPECIALIST_CANDIDATES = {
    "f1_threshold": f1_threshold,
    "recall90_threshold": recall90_threshold,
    "training_classes": ["Exploits", "DoS"],
    "base_classifier": "XGBoost V1",
    "base_decision": "Two-stage V1.1"
}

print("\n" + "=" * 85)
print("CELL 14 COMPLETE")
print("=" * 85)

print("New models trained: 1 specialist")
print("Existing models overwritten: NO")
print("Official TEST accessed: NO")
print("Google Drive modified: NO")
print("Specialist saved permanently: NOT YET")
print("=" * 85)


In [ ]:

# ============================================================
# CELL 15 — SAVE DoS / EXPLOITS SPECIALIST
#
# Experimental model only.
# No retraining.
# No overwriting existing checkpoints.
# No official TEST access.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import numpy as np
import xgboost as xgb

print("=" * 80)
print("CELL 15 — SAVE DoS SPECIALIST")
print("=" * 80)

required = [
    "DOS_SPECIALIST_MODEL",
    "DOS_SPECIALIST_CANDIDATES",
    "X_check_processed",
    "route_mask",
    "routed_scores"
]

for item in required:
    if item not in globals():
        raise RuntimeError(
            f"Missing {item}. Run Cell 14 first."
        )

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

RUN_ID = datetime.now(
    timezone.utc
).strftime("dos_specialist_%Y%m%d_%H%M%S_%f")

SAVE_DIR = (
    PROJECT
    / "Current_Attack_Detection"
    / "specialists"
    / RUN_ID
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=False
)

MODEL_PATH = SAVE_DIR / "dos_exploits_xgboost.ubj"
CONFIG_PATH = SAVE_DIR / "specialist_config.json"

# ------------------------------------------------------------
# 1. Save specialist checkpoint.
# ------------------------------------------------------------

DOS_SPECIALIST_MODEL.save_model(
    str(MODEL_PATH)
)

# ------------------------------------------------------------
# 2. Preserve thresholds and provenance.
# ------------------------------------------------------------

config = {
    "model_name": "NetGuard_DoS_Exploits_Specialist",
    "status": "experimental_not_deployed",

    "task": "DoS_vs_Exploits",

    "base_model": (
        "Current_Attack_Detection/artifacts/"
        "xgboost_20261027_023158_885060"
    ),

    "training_dataset": (
        "UNSW-NB15 canonical training partition"
    ),

    "training_classes": [
        "Exploits",
        "DoS"
    ],

    "positive_class": "DoS",

    "feature_preprocessor": (
        "Reuse the saved current-attack V1 preprocessor"
    ),

    "f1_threshold": float(
        DOS_SPECIALIST_CANDIDATES["f1_threshold"]
    ),

    "high_recall_threshold": (
        float(
            DOS_SPECIALIST_CANDIDATES[
                "recall90_threshold"
            ]
        )
        if DOS_SPECIALIST_CANDIDATES[
            "recall90_threshold"
        ] is not None
        else None
    ),

    "routing": (
        "Experimental V1.1 two-stage predicted "
        "class in {DoS, Exploits}"
    ),

    "evaluation": {
        "type": "exploratory development validation",
        "high_recall_dos_recall": 0.8570,
        "high_recall_dos_precision": 0.3427,
        "high_recall_dos_f1": 0.4896,
        "high_recall_macro_f1": 0.6422,
        "official_test_used": False
    },

    "limitations": [
        "High DoS false-positive burden",
        "Reduced Exploits recall",
        "Not selected for deployment",
        "Requires independent final evaluation"
    ]
}

with CONFIG_PATH.open(
    "x",
    encoding="utf-8"
) as handle:
    json.dump(config, handle, indent=2)

# ------------------------------------------------------------
# 3. Reload checkpoint and verify predictions.
# ------------------------------------------------------------

reloaded_specialist = xgb.XGBClassifier()

reloaded_specialist.load_model(
    str(MODEL_PATH)
)

reloaded_scores = (
    reloaded_specialist.predict_proba(
        X_check_processed[route_mask]
    )[:, 1]
)

np.testing.assert_allclose(
    reloaded_scores,
    routed_scores,
    rtol=1e-6,
    atol=1e-7
)

print("\nSpecialist reload verification: PASS")

# ------------------------------------------------------------
# 4. Generate file integrity hashes.
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

hashes = {
    MODEL_PATH.name: sha256_file(MODEL_PATH),
    CONFIG_PATH.name: sha256_file(CONFIG_PATH)
}

with (
    SAVE_DIR / "artifact_hashes.json"
).open("x", encoding="utf-8") as handle:

    json.dump(hashes, handle, indent=2)

print("\nSaved files:")

for path in sorted(SAVE_DIR.iterdir()):
    print(" -", path.name)

print("\n" + "=" * 80)
print("CELL 15 COMPLETE")
print("=" * 80)

print("Saved directory:", SAVE_DIR)
print("Specialist reload: PASS")
print("New models trained: 0")
print("V1 overwritten: NO")
print("Official TEST accessed: NO")
print("Forecasting models changed: NO")

print("=" * 80)


In [ ]:

# ============================================================
# CELL 16 — DoS HIGH-RECALL THRESHOLD DIAGNOSTIC
#
# Uses the ALREADY TRAINED specialist.
# No retraining.
# Threshold chosen on internal specialist validation.
# Existing check subset used for exploratory comparison.
#
# NO official TEST access.
# NO Drive writes.
# NO existing model modifications.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    precision_recall_curve,
    precision_recall_fscore_support,
    accuracy_score,
    f1_score,
    confusion_matrix
)

print("=" * 80)
print("CELL 16 — DoS HIGH-RECALL DIAGNOSTIC")
print("=" * 80)

required = [
    "DOS_SPECIALIST_MODEL",
    "X_specialist", "y_specialist", "inner_val",
    "X_check_processed", "route_mask",
    "y_check", "baseline", "names",
    "DOS_SPECIALIST_CANDIDATES"
]

for item in required:
    if item not in globals():
        raise RuntimeError(
            f"Missing {item}. Restore Cell 14 outputs."
        )

model16 = DOS_SPECIALIST_MODEL

class_names16 = list(names)
dos_id16 = class_names16.index("DoS")
exp_id16 = class_names16.index("Exploits")
normal_id16 = class_names16.index("Normal")

truth16 = np.asarray(y_check)
base16 = np.asarray(baseline)
route16 = np.asarray(route_mask, dtype=bool)

# ------------------------------------------------------------
# 1. Internal validation: derive threshold for 95% DoS recall.
# ------------------------------------------------------------

internal_prob16 = model16.predict_proba(
    X_specialist[inner_val]
)[:, 1]

internal_truth16 = np.asarray(
    y_specialist
)[inner_val]

precision16, recall16, thresholds16 = (
    precision_recall_curve(
        internal_truth16,
        internal_prob16
    )
)

candidate_indices16 = np.flatnonzero(
    recall16[:-1] >= 0.95
)

if len(candidate_indices16) == 0:
    raise RuntimeError(
        "No internal threshold achieves 95% recall."
    )

best_index16 = max(
    candidate_indices16,
    key=lambda i: (
        float(precision16[i]),
        float(thresholds16[i])
    )
)

threshold95_16 = float(
    thresholds16[best_index16]
)

print("\nA. INTERNAL SPECIALIST VALIDATION")
print("-" * 80)

print(
    "95%-recall target threshold:",
    round(threshold95_16, 6)
)

print(
    "Internal DoS precision:",
    round(float(precision16[best_index16]), 4)
)

print(
    "Internal DoS recall:",
    round(float(recall16[best_index16]), 4)
)

# ------------------------------------------------------------
# 2. Score routed check records once.
# ------------------------------------------------------------

routed_prob16 = model16.predict_proba(
    X_check_processed[route16]
)[:, 1]

thresholds_to_check16 = {
    "Specialist F1 threshold":
        DOS_SPECIALIST_CANDIDATES["f1_threshold"],

    "Specialist 90% internal":
        DOS_SPECIALIST_CANDIDATES["recall90_threshold"],

    "Specialist 95% internal":
        threshold95_16
}

print("\nB. ROUTING LIMITATION")
print("-" * 80)

actual_dos16 = truth16 == dos_id16

routed_dos16 = int(
    (actual_dos16 & route16).sum()
)

total_dos16 = int(actual_dos16.sum())

print("Actual DoS:", total_dos16)
print("DoS routed:", routed_dos16)
print("DoS not routed:", total_dos16 - routed_dos16)

print(
    "Maximum DoS recall with current routing:",
    round(routed_dos16 / total_dos16, 4)
)

# ------------------------------------------------------------
# 3. Compare fixed candidate thresholds.
# ------------------------------------------------------------

results16 = []

def metrics16(prediction):

    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            truth16,
            prediction,
            labels=np.arange(len(class_names16)),
            zero_division=0
        )
    )

    return {
        "Accuracy": accuracy_score(
            truth16, prediction
        ),
        "Macro F1": f1_score(
            truth16, prediction,
            average="macro",
            zero_division=0
        ),
        "DoS precision": precision[dos_id16],
        "DoS recall": recall[dos_id16],
        "DoS F1": f1[dos_id16],
        "Exploits recall": recall[exp_id16]
    }

baseline_metrics16 = metrics16(base16)

results16.append({
    "Method": "V1.1 baseline",
    "Threshold": None,
    **baseline_metrics16
})

for method, threshold in thresholds_to_check16.items():

    if threshold is None:
        continue

    prediction = base16.copy()

    prediction[route16] = np.where(
        routed_prob16 >= float(threshold),
        dos_id16,
        exp_id16
    )

    # Preserve binary Attack/Normal predictions.
    assert np.array_equal(
        prediction != normal_id16,
        base16 != normal_id16
    )

    results16.append({
        "Method": method,
        "Threshold": float(threshold),
        **metrics16(prediction)
    })

results_table16 = pd.DataFrame(results16)

print("\nC. EXPLORATORY COMPARISON")
print("-" * 80)

print(
    results_table16.round(4).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 4. Persist nothing; retain diagnostic table in memory.
# ------------------------------------------------------------

DOS_THRESHOLD_DIAGNOSTICS = results_table16.copy()

print("\n" + "=" * 80)
print("CELL 16 COMPLETE")
print("=" * 80)
print("New models trained: 0")
print("Original model modified: NO")
print("Specialist modified: NO")
print("Official TEST accessed: NO")
print("Drive files modified: NO")
print("=" * 80)


In [ ]:

# ============================================================
# CELL 18 — FULL 10-CLASS CATBOOST EXPERIMENT
#
# Goal: Improve overall classification and all class metrics.
# Trains ONE new model.
#
# Uses existing UNSW development TRAIN / VALIDATION.
# NO official TEST access.
# NO synthetic forecasting data modification.
# NO existing model overwrites.
# ============================================================

import sys
import subprocess
import time

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    precision_recall_fscore_support,
    average_precision_score
)

try:
    from catboost import CatBoostClassifier
except ImportError:
    print("Installing CatBoost...")
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install",
            "-q", "catboost==1.2.10"
        ],
        check=True
    )
    from catboost import CatBoostClassifier

print("=" * 85)
print("CELL 18 — FULL 10-CLASS CATBOOST")
print("=" * 85)

required = [
    "X_train", "X_val",
    "y_train", "y_val",
    "CLASS_NAMES", "proba", "yva"
]

for item in required:
    if item not in globals():
        raise RuntimeError(
            f"Missing {item}. Restore the earlier split "
            "and XGBoost validation outputs first."
        )

# ------------------------------------------------------------
# 1. Use exactly the same features and split as XGBoost.
# ------------------------------------------------------------

cat_cols18 = [
    c for c in ["proto", "service", "state"]
    if c in X_train.columns
]

num_cols18 = [
    c for c in X_train.columns
    if c not in cat_cols18
]

feature_order18 = X_train.columns.tolist()

def prepare_catboost_data(frame):

    result = frame[feature_order18].copy()

    for col in cat_cols18:
        result[col] = (
            result[col]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    for col in num_cols18:
        result[col] = pd.to_numeric(
            result[col],
            errors="coerce"
        ).replace(
            [np.inf, -np.inf],
            np.nan
        )

    return result

Xtr18 = prepare_catboost_data(X_train)
Xva18 = prepare_catboost_data(X_val)

ytr18 = y_train.astype(str).str.strip()
yva18 = y_val.astype(str).str.strip()

assert list(Xtr18.columns) == list(Xva18.columns)
assert set(ytr18.unique()) == set(CLASS_NAMES)

# Verify XGBoost reference labels are aligned.
assert np.array_equal(
    yva18.to_numpy(),
    np.asarray(CLASS_NAMES)[np.asarray(yva)]
)

print("\nTraining rows:", len(Xtr18))
print("Validation rows:", len(Xva18))
print("Features:", Xtr18.shape[1])
print("Categorical:", cat_cols18)
print("Classes:", sorted(ytr18.unique()))

# ------------------------------------------------------------
# 2. Train exactly ONE CatBoost model.
#
# No class weighting initially: establish whether a
# different learning algorithm improves all-class performance.
#
# No validation-based early stopping in this run.
# ------------------------------------------------------------

model18 = CatBoostClassifier(
    loss_function="MultiClass",
    iterations=500,
    depth=7,
    learning_rate=0.07,
    l2_leaf_reg=5,
    random_seed=42,
    thread_count=4,
    task_type="CPU",
    allow_writing_files=False,
    verbose=100
)

print("\nTraining CatBoost...")
start18 = time.time()

model18.fit(
    Xtr18,
    ytr18,
    cat_features=cat_cols18
)

print(
    "\nTraining seconds:",
    round(time.time() - start18, 2)
)

# ------------------------------------------------------------
# 3. Evaluate CatBoost.
# ------------------------------------------------------------

pred18 = np.asarray(
    model18.predict(Xva18)
).ravel().astype(str)

cb_classes18 = [
    str(c) for c in model18.classes_
]

prob18 = model18.predict_proba(Xva18)

assert prob18.shape == (
    len(Xva18), len(CLASS_NAMES)
)

# ------------------------------------------------------------
# 4. Compare to original XGBoost on identical validation.
# ------------------------------------------------------------

original_xgb_pred18 = np.asarray(
    CLASS_NAMES
)[
    np.argmax(
        np.asarray(proba),
        axis=1
    )
]

def calculate18(predictions):

    return {
        "Accuracy": accuracy_score(
            yva18, predictions
        ),
        "Balanced accuracy": balanced_accuracy_score(
            yva18, predictions
        ),
        "Macro F1": f1_score(
            yva18, predictions,
            average="macro",
            zero_division=0
        ),
        "Weighted F1": f1_score(
            yva18, predictions,
            average="weighted",
            zero_division=0
        )
    }

xgb_metrics18 = calculate18(
    original_xgb_pred18
)

cb_metrics18 = calculate18(pred18)

print("\n" + "=" * 85)
print("OVERALL MODEL COMPARISON")
print("=" * 85)

for metric in xgb_metrics18:
    old = xgb_metrics18[metric]
    new = cb_metrics18[metric]

    print(
        f"{metric:<22} "
        f"XGBoost={old:.4f} "
        f"CatBoost={new:.4f} "
        f"Change={new-old:+.4f}"
    )

# ------------------------------------------------------------
# 5. Compare every single class.
# ------------------------------------------------------------

class_order18 = list(CLASS_NAMES)

old_p, old_r, old_f, old_n = (
    precision_recall_fscore_support(
        yva18,
        original_xgb_pred18,
        labels=class_order18,
        zero_division=0
    )
)

new_p, new_r, new_f, new_n = (
    precision_recall_fscore_support(
        yva18,
        pred18,
        labels=class_order18,
        zero_division=0
    )
)

comparison18 = pd.DataFrame({
    "Class": class_order18,
    "Support": new_n,
    "XGB Recall": old_r,
    "CAT Recall": new_r,
    "CAT Precision": new_p,
    "XGB F1": old_f,
    "CAT F1": new_f
})

print("\n" + "=" * 85)
print("EVERY CLASS — BEFORE VS AFTER")
print("=" * 85)

print(
    comparison18.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 6. Calculate multiclass AP for CatBoost.
# ------------------------------------------------------------

cat_ap18 = []

for class_name in class_order18:

    class_index = cb_classes18.index(class_name)

    class_truth = (
        yva18.to_numpy() == class_name
    ).astype(int)

    ap = average_precision_score(
        class_truth,
        prob18[:, class_index]
    )

    cat_ap18.append(ap)

print("\n" + "=" * 85)
print("CATBOOST ONE-VS-REST AVERAGE PRECISION")
print("=" * 85)

for name, ap in zip(class_order18, cat_ap18):
    print(f"{name:<18}: {ap:.4f}")

print(
    "\nMacro AP:",
    round(float(np.mean(cat_ap18)), 4)
)

# ------------------------------------------------------------
# 7. Explicitly check the user's 90% goals.
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("90% TARGET CHECK")
print("=" * 85)

overall_target18 = (
    cb_metrics18["Accuracy"] >= 0.90
)

class_target18 = new_r >= 0.90

print(
    "Overall accuracy >= 90%:",
    overall_target18
)

print(
    "Classes with recall >= 90%:",
    int(class_target18.sum()),
    "of",
    len(class_order18)
)

print(
    "Minimum class recall:",
    round(float(new_r.min()), 4)
)

print(
    "Classes below 90% recall:",
    [
        class_order18[i]
        for i in range(len(class_order18))
        if not class_target18[i]
    ]
)

# ------------------------------------------------------------
# 8. Preserve in memory. Don't replace saved XGBoost.
# ------------------------------------------------------------

CATBOOST_MODEL18 = model18
CATBOOST_FEATURES18 = feature_order18
CATBOOST_CAT_COLS18 = cat_cols18
CATBOOST_METRICS18 = cb_metrics18
CATBOOST_COMPARISON18 = comparison18

print("\n" + "=" * 85)
print("CELL 18 COMPLETE")
print("=" * 85)
print("New models trained: 1")
print("Existing saved models overwritten: NO")
print("Official TEST accessed: NO")
print("Forecasting models modified: NO")
print("New model saved to Drive: NOT YET")
print("=" * 85)


In [ ]:

# ============================================================
# CELL 19 — PRESERVE CATBOOST + MEASURE LABEL AMBIGUITY
#
# NO retraining
# NO official TEST access
# NO existing model overwrites
# Creates a NEW CatBoost experiment directory
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

print("=" * 80)
print("CELL 19 — SAVE EXPERIMENT AND CHECK ACCURACY LIMIT")
print("=" * 80)

required = [
    "CATBOOST_MODEL18",
    "CATBOOST_FEATURES18",
    "CATBOOST_CAT_COLS18",
    "CATBOOST_METRICS18",
    "Xva18",
    "groups",
    "val_idx",
    "y_val"
]

for item in required:
    if item not in globals():
        raise RuntimeError(f"Missing {item}")

# ------------------------------------------------------------
# A. Preserve the CatBoost experiment.
# ------------------------------------------------------------

PROJECT = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

EXPERIMENT_ROOT = (
    PROJECT
    / "Current_Attack_Detection"
    / "experiments"
)

run_id = datetime.now(
    timezone.utc
).strftime("catboost_%Y%m%d_%H%M%S_%f")

experiment_dir = EXPERIMENT_ROOT / run_id

experiment_dir.mkdir(
    parents=True,
    exist_ok=False
)

model_path = (
    experiment_dir / "catboost_10class.cbm"
)

CATBOOST_MODEL18.save_model(str(model_path))

metadata = {
    "status": "experimental_not_selected",
    "task": "current_attack_10class",
    "dataset": "UNSW-NB15 canonical TRAIN",
    "features": list(CATBOOST_FEATURES18),
    "categorical_features": list(CATBOOST_CAT_COLS18),
    "class_names": [
        str(x) for x in CATBOOST_MODEL18.classes_
    ],
    "metrics": {
        key: float(value)
        for key, value in CATBOOST_METRICS18.items()
    },
    "official_test_used": False,
    "input_preparation": (
        "Categoricals converted to strings with missing "
        "value __MISSING__; numeric columns converted "
        "with pd.to_numeric and infinity replaced by NaN"
    )
}

with (
    experiment_dir / "metadata.json"
).open("x", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# Verify reload without retraining.
reloaded = CatBoostClassifier()
reloaded.load_model(str(model_path))

sample = Xva18.iloc[:50]

original_scores = CATBOOST_MODEL18.predict_proba(
    sample
)

reloaded_scores = reloaded.predict_proba(
    sample
)

np.testing.assert_allclose(
    original_scores,
    reloaded_scores,
    rtol=1e-8,
    atol=1e-8
)

print("\nA. CATBOOST ARTIFACT")
print("-" * 80)
print("Saved:", experiment_dir)
print("Reload verification: PASS")
print("Main XGBoost model: UNCHANGED")

# ------------------------------------------------------------
# B. Calculate label-conflict accuracy ceiling.
#
# For each distinct feature group, a deterministic
# classifier can predict only one class.
#
# Best possible correct predictions per group =
# frequency of that group's most common label.
#
# This uses validation labels to compute an OPTIMISTIC
# theoretical ceiling, not a trainable prediction rule.
# ------------------------------------------------------------

validation_hashes = (
    groups.iloc[val_idx].to_numpy()
)

validation_labels = (
    y_val.astype(str).str.strip().to_numpy()
)

assert len(validation_hashes) == len(validation_labels)

audit = pd.DataFrame({
    "feature_group": validation_hashes,
    "true_class": validation_labels
})

group_label_counts = (
    audit
    .groupby(
        ["feature_group", "true_class"]
    )
    .size()
    .rename("count")
    .reset_index()
)

group_sizes = (
    audit
    .groupby("feature_group")
    .size()
)

group_best_possible = (
    group_label_counts
    .groupby("feature_group")["count"]
    .max()
)

group_label_diversity = (
    group_label_counts
    .groupby("feature_group")["true_class"]
    .nunique()
)

ambiguous_groups = group_label_diversity[
    group_label_diversity > 1
].index

ambiguous_records = int(
    group_sizes.loc[ambiguous_groups].sum()
)

max_correct = int(group_best_possible.sum())

minimum_unavoidable_errors = (
    len(audit) - max_correct
)

accuracy_ceiling = max_correct / len(audit)

print("\nB. VALIDATION FEATURE/LABEL AMBIGUITY")
print("-" * 80)

print("Validation records:", len(audit))
print("Unique feature groups:", len(group_sizes))
print("Conflicting-label groups:", len(ambiguous_groups))
print("Records in conflicting groups:", ambiguous_records)

print(
    "Minimum errors from identical-feature "
    "label conflicts:",
    minimum_unavoidable_errors
)

print(
    "Optimistic deterministic accuracy ceiling:",
    f"{accuracy_ceiling:.2%}"
)

print(
    "Current XGBoost accuracy:",
    f"{CATBOOST_METRICS18['Accuracy'] + 0.0079:.2%}",
    "(approximate previous baseline)"
)

print("\nC. INTERPRETATION")
print("-" * 80)

if accuracy_ceiling < 0.90:
    print(
        "A deterministic classifier using only these "
        "feature vectors cannot reach 90% accuracy "
        "on this validation set, even with perfect "
        "knowledge of the group-majority labels."
    )
else:
    print(
        "The observed duplicate-label conflicts do "
        "not mathematically rule out 90% accuracy. "
        "However, reaching it requires better "
        "generalization than our current models."
    )

print(
    "\nNote: This is an optimistic in-sample ceiling, "
    "NOT an achievable test-accuracy estimate."
)

print("\n" + "=" * 80)
print("CELL 19 COMPLETE")
print("=" * 80)
print("CatBoost saved: YES")
print("New training: NO")
print("Official TEST accessed: NO")
print("Forecasting models changed: NO")
print("=" * 80)


In [ ]:

# ============================================================
# CELL 20 — CLASS-SPECIFIC LABEL CONFLICT ANALYSIS
#
# Diagnose why individual attack types fail.
# No training, no TEST access, no file writes.
# ============================================================

import numpy as np
import pandas as pd
from collections import Counter
from itertools import combinations

print("=" * 85)
print("CELL 20 — CLASS-SPECIFIC ERROR DIAGNOSIS")
print("=" * 85)

required = [
    "groups", "val_idx",
    "y_val", "proba", "CLASS_NAMES"
]

for name in required:
    if name not in globals():
        raise RuntimeError(f"Missing {name}")

# ------------------------------------------------------------
# 1. Reconstruct validation records and predictions.
# ------------------------------------------------------------

labels20 = (
    y_val.astype(str).str.strip().to_numpy()
)

hashes20 = groups.iloc[val_idx].to_numpy()

predictions20 = np.asarray(CLASS_NAMES)[
    np.argmax(np.asarray(proba), axis=1)
]

assert (
    len(labels20)
    == len(hashes20)
    == len(predictions20)
)

records20 = pd.DataFrame({
    "feature_group": hashes20,
    "actual": labels20,
    "predicted": predictions20
})

records20["correct"] = (
    records20["actual"]
    == records20["predicted"]
)

# ------------------------------------------------------------
# 2. Identify conflicting-label feature groups.
# ------------------------------------------------------------

diversity20 = (
    records20.groupby("feature_group")["actual"]
    .nunique()
)

conflict_ids20 = set(
    diversity20[diversity20 > 1].index
)

records20["ambiguous"] = (
    records20["feature_group"]
    .isin(conflict_ids20)
)

print("\nA. OVERALL ERROR DECOMPOSITION")
print("-" * 85)

for title, subset in [
    ("Unambiguous", records20[~records20["ambiguous"]]),
    ("Ambiguous", records20[records20["ambiguous"]])
]:
    errors = int((~subset["correct"]).sum())

    print(
        f"{title:<15} "
        f"Rows={len(subset):>6} "
        f"Errors={errors:>6} "
        f"Accuracy={subset['correct'].mean():.4f}"
    )

# ------------------------------------------------------------
# 3. How ambiguity affects EACH class.
# ------------------------------------------------------------

class_results20 = []

for class_name in CLASS_NAMES:

    subset = records20[
        records20["actual"] == class_name
    ]

    ambiguous = subset["ambiguous"]

    errors = ~subset["correct"]

    class_results20.append({
        "Class": class_name,
        "Total": int(len(subset)),
        "Ambiguous": int(ambiguous.sum()),
        "Ambiguous %": float(
            100 * ambiguous.mean()
        ),
        "All errors": int(errors.sum()),
        "Errors in ambiguous": int(
            (errors & ambiguous).sum()
        ),
        "Errors in unambiguous": int(
            (errors & ~ambiguous).sum()
        ),
        "Recall": float(subset["correct"].mean())
    })

class_table20 = pd.DataFrame(class_results20)

print("\nB. CLASS-WISE LABEL CONFLICTS AND ERRORS")
print("-" * 85)

print(
    class_table20.round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 4. Find most common conflicting-label pairs.
#
# Reports how often two different labels occur
# for the same complete feature vector.
# ------------------------------------------------------------

conflicting20 = records20[
    records20["ambiguous"]
]

pair_counts20 = Counter()

for _, labels in (
    conflicting20
    .groupby("feature_group")["actual"]
    .unique()
    .items()
):

    distinct_labels = sorted(
        set(labels)
    )

    for pair in combinations(
        distinct_labels, 2
    ):
        pair_counts20[pair] += 1

print("\nC. MOST COMMON CONFLICTING LABEL PAIRS")
print("-" * 85)

for pair, count in pair_counts20.most_common(12):

    print(
        f"{pair[0]:<18} "
        f"<-> {pair[1]:<18} "
        f"Groups={count}"
    )

# ------------------------------------------------------------
# 5. Calculate the label-conflict error floor
#    and remaining theoretical headroom.
# ------------------------------------------------------------

counts20 = (
    records20
    .groupby(["feature_group", "actual"])
    .size()
)

majority20 = (
    counts20
    .groupby(level=0)
    .max()
)

oracle_correct20 = int(
    majority20.sum()
)

conflict_error_floor20 = (
    len(records20) - oracle_correct20
)

model_errors20 = int(
    (~records20["correct"]).sum()
)

print("\nD. THEORETICAL LIMIT")
print("-" * 85)

print(
    "Original XGBoost errors:",
    model_errors20
)

print(
    "Minimum errors from conflicting labels:",
    conflict_error_floor20
)

print(
    "Additional errors beyond conflict floor:",
    model_errors20 - conflict_error_floor20
)

print(
    "Optimistic accuracy ceiling:",
    round(
        oracle_correct20 / len(records20),
        4
    )
)

print("\n" + "=" * 85)
print("CELL 20 COMPLETE")
print("=" * 85)

print("New models trained: 0")
print("Official TEST accessed: NO")
print("Existing checkpoints modified: NO")
print("Files written: NO")

print("=" * 85)


In [ ]:

# ============================================================
# CELL 21 — INSPECT AMBIGUOUS NETWORK FLOW SIGNATURES
#
# No training, no TEST access, no file modifications.
# ============================================================

import numpy as np
import pandas as pd

print("=" * 82)
print("CELL 21 — CONFLICTING NETWORK FLOW SIGNATURES")
print("=" * 82)

required = ["X_val", "y_val", "groups", "val_idx"]

for key in required:
    if key not in globals():
        raise RuntimeError(f"Missing {key}. Restore Cell 6.")

# Construct read-only validation view.
flows21 = X_val.reset_index(drop=True).copy()

flows21["_group"] = (
    groups.iloc[val_idx].to_numpy()
)

flows21["_class"] = (
    y_val.astype(str).str.strip().to_numpy()
)

counts21 = (
    flows21
    .groupby(["_group", "_class"])
    .size()
    .unstack(fill_value=0)
)

group_sizes21 = counts21.sum(axis=1)
label_diversity21 = (counts21 > 0).sum(axis=1)

conflicts21 = counts21[
    label_diversity21 > 1
].copy()

conflicts21["_total"] = (
    conflicts21.sum(axis=1)
)

conflicts21["_minimum_errors"] = (
    conflicts21["_total"]
    - conflicts21.drop(
        columns="_total"
    ).max(axis=1)
)

conflicts21 = conflicts21.sort_values(
    ["_minimum_errors", "_total"],
    ascending=False
)

print("\nA. WORST CONFLICTING FEATURE GROUPS")
print("-" * 82)

display_cols21 = [
    c for c in [
        "proto", "service", "state",
        "dur", "sbytes", "dbytes",
        "sttl", "dttl",
        "ct_state_ttl",
        "ct_srv_dst",
        "ct_dst_src_ltm"
    ]
    if c in X_val.columns
]

for rank, (gid, row) in enumerate(
    conflicts21.head(6).iterrows(),
    start=1
):

    distribution = {
        str(name): int(value)
        for name, value in row.items()
        if name not in [
            "_total", "_minimum_errors"
        ] and value > 0
    }

    example = flows21.loc[
        flows21["_group"] == gid,
        display_cols21
    ].iloc[0]

    print(f"\nCONFLICT GROUP {rank}")
    print("Records:", int(row["_total"]))
    print(
        "Minimum unavoidable errors:",
        int(row["_minimum_errors"])
    )
    print("Conflicting labels:", distribution)
    print("Shared traffic features:")

    print(example.to_string())

# ------------------------------------------------------------
# B. Class recall on non-conflicting signatures.
# ------------------------------------------------------------

print("\nB. AMBIGUITY-FREE SUBSET SIZE")
print("-" * 82)

ambiguous_ids21 = set(conflicts21.index)

flows21["_ambiguous"] = (
    flows21["_group"].isin(ambiguous_ids21)
)

for name in sorted(flows21["_class"].unique()):

    subset = flows21[
        flows21["_class"] == name
    ]

    clean_count = int(
        (~subset["_ambiguous"]).sum()
    )

    ambiguous_count = int(
        subset["_ambiguous"].sum()
    )

    print(
        f"{name:<18} "
        f"Non-conflicting={clean_count:>6} "
        f"Conflicting={ambiguous_count:>6}"
    )

# ------------------------------------------------------------
# C. How concentrated are the conflicting errors?
# ------------------------------------------------------------

print("\nC. CONFLICT CONCENTRATION")
print("-" * 82)

total_minimum_errors21 = int(
    conflicts21["_minimum_errors"].sum()
)

for k in [5, 10, 25, 50, 100]:

    top_errors = int(
        conflicts21["_minimum_errors"]
        .head(k)
        .sum()
    )

    print(
        f"Top {k:>3} groups: "
        f"{top_errors:>5} minimum errors "
        f"({top_errors / total_minimum_errors21:.1%} "
        "of conflict error floor)"
    )

print("\n" + "=" * 82)
print("CELL 21 COMPLETE")
print("=" * 82)

print("Models trained: 0")
print("Official TEST accessed: NO")
print("Saved checkpoints modified: NO")
print("Files written: NO")


In [ ]:

# ============================================================
# CELL 22 — CLASS-WEIGHTED 10-CLASS XGBOOST
#
# One training experiment.
# Mild class weighting, NO synthetic samples.
# Uses existing train-only fitted preprocessing.
#
# NO official TEST access.
# NO Drive writes.
# NO modifications to existing models.
# ============================================================

import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    average_precision_score
)

print("=" * 85)
print("CELL 22 — CLASS-WEIGHTED XGBOOST")
print("=" * 85)

required = [
    "Xtr_processed",
    "Xva_processed",
    "ytr",
    "yva",
    "proba",
    "CLASS_NAMES"
]

for key in required:
    if key not in globals():
        raise RuntimeError(
            f"Missing {key}. Restore earlier training outputs."
        )

classes22 = list(CLASS_NAMES)

y_train22 = np.asarray(ytr, dtype=int)
y_val22 = np.asarray(yva, dtype=int)

p_baseline22 = np.asarray(proba)

assert p_baseline22.shape == (
    len(y_val22),
    len(classes22)
)

# ------------------------------------------------------------
# 1. Calculate mild inverse-frequency class weights.
#
# More moderate than full inverse class frequency.
# Maximum relative weight capped at 3.
#
# Uses TRAIN labels only.
# ------------------------------------------------------------

counts22 = np.bincount(
    y_train22,
    minlength=len(classes22)
)

assert np.all(counts22 > 0)

largest_count22 = counts22.max()

class_weights22 = np.minimum(
    (largest_count22 / counts22) ** 0.35,
    3.0
)

sample_weights22 = class_weights22[y_train22]

# Normalize average weight to 1 so overall scale
# remains comparable to the original model.
sample_weights22 = (
    sample_weights22 /
    sample_weights22.mean()
)

print("\nA. TRAINING CLASS WEIGHTS")
print("-" * 85)

for i, name in enumerate(classes22):
    print(
        f"{name:<18} "
        f"rows={counts22[i]:>6} "
        f"relative_weight={class_weights22[i]:.3f}"
    )

print(
    "\nMean normalized sample weight:",
    round(float(sample_weights22.mean()), 4)
)

# ------------------------------------------------------------
# 2. Train ONE weighted XGBoost.
#
# Keep the same architecture as our V1 baseline
# to isolate the class-weighting effect.
# ------------------------------------------------------------

model22 = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(classes22),

    n_estimators=180,
    max_depth=6,
    learning_rate=0.10,

    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,

    tree_method="hist",
    n_jobs=4,
    random_state=42,

    eval_metric="mlogloss"
)

print("\nB. TRAINING")
print("-" * 85)

start22 = time.time()

model22.fit(
    Xtr_processed,
    y_train22,
    sample_weight=sample_weights22,
    verbose=False
)

print(
    "Training seconds:",
    round(time.time() - start22, 2)
)

# ------------------------------------------------------------
# 3. Predict without any post-hoc decision weights.
# ------------------------------------------------------------

p_weighted22 = model22.predict_proba(
    Xva_processed
)

pred_weighted22 = np.argmax(
    p_weighted22,
    axis=1
)

pred_baseline22 = np.argmax(
    p_baseline22,
    axis=1
)

# ------------------------------------------------------------
# 4. Compare full-model performance.
# ------------------------------------------------------------

def metrics22(pred):

    return {
        "Accuracy": accuracy_score(
            y_val22, pred
        ),

        "Macro F1": f1_score(
            y_val22,
            pred,
            average="macro",
            zero_division=0
        ),

        "Balanced accuracy": balanced_accuracy_score(
            y_val22, pred
        ),

        "Weighted F1": f1_score(
            y_val22,
            pred,
            average="weighted",
            zero_division=0
        )
    }

old_metrics22 = metrics22(
    pred_baseline22
)

new_metrics22 = metrics22(
    pred_weighted22
)

print("\n" + "=" * 85)
print("C. OVERALL PERFORMANCE")
print("=" * 85)

for metric in old_metrics22:

    old = old_metrics22[metric]
    new = new_metrics22[metric]

    print(
        f"{metric:<22} "
        f"V1={old:.4f} "
        f"Weighted={new:.4f} "
        f"Change={new-old:+.4f}"
    )

# ------------------------------------------------------------
# 5. Per-class performance.
# ------------------------------------------------------------

labels22 = np.arange(
    len(classes22)
)

old_p22, old_r22, old_f22, support22 = (
    precision_recall_fscore_support(
        y_val22,
        pred_baseline22,
        labels=labels22,
        zero_division=0
    )
)

new_p22, new_r22, new_f22, _ = (
    precision_recall_fscore_support(
        y_val22,
        pred_weighted22,
        labels=labels22,
        zero_division=0
    )
)

comparison22 = pd.DataFrame({
    "Class": classes22,
    "Support": support22,
    "V1 recall": old_r22,
    "Weighted recall": new_r22,
    "V1 F1": old_f22,
    "Weighted F1": new_f22,
    "Weighted precision": new_p22
})

print("\n" + "=" * 85)
print("D. ALL TEN CLASSES")
print("=" * 85)

print(
    comparison22.round(4).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 6. Compare probability ranking (AP).
# ------------------------------------------------------------

ap_rows22 = []

for i, name in enumerate(classes22):

    truth_binary = (
        y_val22 == i
    ).astype(int)

    old_ap = average_precision_score(
        truth_binary,
        p_baseline22[:, i]
    )

    new_ap = average_precision_score(
        truth_binary,
        p_weighted22[:, i]
    )

    ap_rows22.append({
        "Class": name,
        "V1 AP": old_ap,
        "Weighted AP": new_ap,
        "Change": new_ap - old_ap
    })

ap_table22 = pd.DataFrame(
    ap_rows22
)

print("\n" + "=" * 85)
print("E. PER-CLASS AVERAGE PRECISION")
print("=" * 85)

print(
    ap_table22.round(4).to_string(
        index=False
    )
)

print(
    "\nMacro AP:",
    f"V1={ap_table22['V1 AP'].mean():.4f}, "
    f"Weighted={ap_table22['Weighted AP'].mean():.4f}"
)

# ------------------------------------------------------------
# 7. Check against our goals.
# ------------------------------------------------------------

print("\n" + "=" * 85)
print("F. TARGET CHECK")
print("=" * 85)

print(
    "Overall accuracy >= 90%:",
    new_metrics22["Accuracy"] >= 0.90
)

print(
    "Macro F1 improved:",
    new_metrics22["Macro F1"] >
    old_metrics22["Macro F1"]
)

print(
    "Classes with recall >= 90%:",
    int(np.sum(new_r22 >= 0.90)),
    "of",
    len(classes22)
)

print(
    "Minimum class recall:",
    round(float(new_r22.min()), 4)
)

# ------------------------------------------------------------
# 8. Keep experimental model in memory.
# ------------------------------------------------------------

WEIGHTED_XGB_MODEL22 = model22
WEIGHTED_XGB_PROBA22 = p_weighted22
WEIGHTED_XGB_METRICS22 = new_metrics22
WEIGHTED_XGB_CLASS_WEIGHTS22 = class_weights22

print("\n" + "=" * 85)
print("CELL 22 COMPLETE")
print("=" * 85)

print("Models trained: 1")
print("Synthetic oversampling used: NO")
print("Official TEST accessed: NO")
print("Saved checkpoints modified: NO")
print("Drive files modified: NO")
print("New checkpoint saved: NOT YET")

print("=" * 85)


In [ ]:

# ============================================================
# CELL 23 — SAVE WEIGHTED XGBOOST V2
#
# No retraining.
# No official TEST access.
# Does NOT overwrite V1 or forecasting models.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import joblib
import numpy as np
import xgboost as xgb

print("=" * 80)
print("CELL 23 — SAVE CLASS-WEIGHTED XGBOOST V2")
print("=" * 80)

required23 = [
    "WEIGHTED_XGB_MODEL22",
    "WEIGHTED_XGB_PROBA22",
    "WEIGHTED_XGB_METRICS22",
    "WEIGHTED_XGB_CLASS_WEIGHTS22",
    "CURRENT_ATTACK_PREPROCESSOR",
    "CURRENT_ATTACK_FEATURE_COLUMNS",
    "CATEGORICAL_COLS",
    "NUMERICAL_COLS",
    "CLASS_NAMES",
    "Xva_processed",
    "y_val"
]

for name in required23:
    if name not in globals():
        raise RuntimeError(f"Missing {name}")

PROJECT23 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

RUN_ID23 = datetime.now(
    timezone.utc
).strftime("weighted_xgb_v2_%Y%m%d_%H%M%S_%f")

SAVE_DIR23 = (
    PROJECT23
    / "Current_Attack_Detection"
    / "experiments"
    / RUN_ID23
)

SAVE_DIR23.mkdir(parents=True, exist_ok=False)

model_path23 = SAVE_DIR23 / "weighted_xgboost.ubj"
prep_path23 = SAVE_DIR23 / "preprocessor.joblib"
metadata_path23 = SAVE_DIR23 / "metadata.json"

# ------------------------------------------------------------
# 1. Save model and fitted preprocessing.
# ------------------------------------------------------------

WEIGHTED_XGB_MODEL22.save_model(
    str(model_path23)
)

joblib.dump(
    CURRENT_ATTACK_PREPROCESSOR,
    prep_path23
)

metadata23 = {
    "name": "NetGuard_Current_Attack_Weighted_XGBoost_V2",
    "status": "experimental_not_deployed",

    "dataset": "UNSW-NB15 canonical TRAIN partition",
    "official_test_used": False,

    "classes": list(CLASS_NAMES),
    "feature_columns": list(
        CURRENT_ATTACK_FEATURE_COLUMNS
    ),
    "categorical_columns": list(
        CATEGORICAL_COLS
    ),
    "numerical_columns": list(
        NUMERICAL_COLS
    ),

    "input_preparation": {
        "categorical": (
            "Convert to pandas string, replace missing "
            "with __MISSING__, then convert to str"
        ),
        "numerical": (
            "pd.to_numeric(errors='coerce'); "
            "replace infinities with NaN"
        )
    },

    "class_weights_before_sample_normalization": {
        name: float(weight)
        for name, weight in zip(
            CLASS_NAMES,
            WEIGHTED_XGB_CLASS_WEIGHTS22
        )
    },

    "validation_metrics": {
        key: float(value)
        for key, value in
        WEIGHTED_XGB_METRICS22.items()
    },

    "evaluation_note": (
        "Exploratory development-validation results. "
        "This validation set was used for multiple "
        "model comparisons; no independent final "
        "performance estimate is claimed."
    )
}

with metadata_path23.open(
    "x", encoding="utf-8"
) as handle:
    json.dump(metadata23, handle, indent=2)

# ------------------------------------------------------------
# 2. Verify saved model reproduces original predictions.
# ------------------------------------------------------------

reloaded23 = xgb.XGBClassifier()

reloaded23.load_model(str(model_path23))

p_reloaded23 = reloaded23.predict_proba(
    Xva_processed
)

np.testing.assert_allclose(
    p_reloaded23,
    WEIGHTED_XGB_PROBA22,
    rtol=1e-6,
    atol=1e-7
)

print("\nModel reload verification: PASS")

# ------------------------------------------------------------
# 3. Generate integrity hashes.
# ------------------------------------------------------------

def sha256_23(path):

    digest = hashlib.sha256()

    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

hashes23 = {
    path.name: sha256_23(path)
    for path in [
        model_path23,
        prep_path23,
        metadata_path23
    ]
}

with (
    SAVE_DIR23 / "artifact_hashes.json"
).open("x", encoding="utf-8") as handle:

    json.dump(hashes23, handle, indent=2)

print("\nSaved files:")

for path in sorted(SAVE_DIR23.iterdir()):
    print(" -", path.name)

print("\n" + "=" * 80)
print("CELL 23 COMPLETE")
print("=" * 80)

print("Saved directory:", SAVE_DIR23)
print("V2 reload verification: PASS")
print("Models retrained: 0")
print("Original XGBoost V1 overwritten: NO")
print("Official TEST accessed: NO")
print("Forecasting models modified: NO")
print("=" * 80)


In [ ]:

# ============================================================
# NETGUARD CURRENT ATTACK DETECTION
# CELL 24 — XGBOOST V1 + WEIGHTED V2 ENSEMBLE
#
# NO training.
# NO new preprocessing.
# NO official TEST access.
# NO checkpoint modifications.
#
# Exploratory validation only.
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    average_precision_score,
    confusion_matrix
)

print("=" * 85)
print("CELL 24 — V1 + V2 PROBABILITY ENSEMBLE")
print("=" * 85)

required = [
    "proba",
    "WEIGHTED_XGB_PROBA22",
    "yva",
    "CLASS_NAMES"
]

for name in required:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Restore Cells 7 and 22."
        )

P_V1 = np.asarray(proba, dtype=float)
P_V2 = np.asarray(
    WEIGHTED_XGB_PROBA22,
    dtype=float
)

Y24 = np.asarray(yva, dtype=int)
NAMES24 = list(CLASS_NAMES)

assert P_V1.shape == P_V2.shape
assert P_V1.shape == (len(Y24), len(NAMES24))

assert np.all(np.isfinite(P_V1))
assert np.all(np.isfinite(P_V2))

np.testing.assert_allclose(
    P_V1.sum(axis=1),
    1.0,
    atol=1e-5
)

np.testing.assert_allclose(
    P_V2.sum(axis=1),
    1.0,
    atol=1e-5
)

# ------------------------------------------------------------
# 1. Fixed ensemble ratios.
#
# alpha = V1 contribution
# (1 - alpha) = V2 contribution
#
# No model fitting or calibration performed.
# ------------------------------------------------------------

ALPHAS24 = [0.0, 0.25, 0.50, 0.75, 1.0]

def blend24(alpha):
    return (
        alpha * P_V1
        + (1.0 - alpha) * P_V2
    )

def evaluate24(probabilities):

    predictions = np.argmax(
        probabilities,
        axis=1
    )

    return {
        "Accuracy": accuracy_score(
            Y24, predictions
        ),
        "Macro F1": f1_score(
            Y24,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Balanced accuracy":
            balanced_accuracy_score(
                Y24, predictions
            ),
        "Weighted F1": f1_score(
            Y24,
            predictions,
            average="weighted",
            zero_division=0
        )
    }

rows24 = []

for alpha in ALPHAS24:

    probabilities = blend24(alpha)

    metrics = evaluate24(
        probabilities
    )

    rows24.append({
        "V1 weight": alpha,
        "V2 weight": 1 - alpha,
        **metrics
    })

RESULTS24 = pd.DataFrame(rows24)

print("\nA. OVERALL ENSEMBLE COMPARISON")
print("-" * 85)

print(
    RESULTS24.round(4).to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 2. Choose an exploratory candidate based on macro F1.
#
# NOTE: This is NOT an unbiased performance estimate.
# We are examining a previously used validation set.
# ------------------------------------------------------------

best_idx24 = (
    RESULTS24["Macro F1"].idxmax()
)

BEST_ALPHA24 = float(
    RESULTS24.loc[
        best_idx24, "V1 weight"
    ]
)

P_BEST24 = blend24(
    BEST_ALPHA24
)

PRED_BEST24 = np.argmax(
    P_BEST24,
    axis=1
)

PRED_V1_24 = np.argmax(
    P_V1,
    axis=1
)

PRED_V2_24 = np.argmax(
    P_V2,
    axis=1
)

print("\nB. EXPLORATORY BEST MACRO-F1 BLEND")
print("-" * 85)

print("V1 contribution:", BEST_ALPHA24)
print("V2 contribution:", 1 - BEST_ALPHA24)

print(
    "Accuracy:",
    round(
        accuracy_score(Y24, PRED_BEST24),
        4
    )
)

print(
    "Macro F1:",
    round(
        f1_score(
            Y24,
            PRED_BEST24,
            average="macro",
            zero_division=0
        ),
        4
    )
)

# ------------------------------------------------------------
# 3. Per-class recall, precision and F1.
# ------------------------------------------------------------

class_ids24 = np.arange(
    len(NAMES24)
)

def class_metrics24(pred):

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            Y24,
            pred,
            labels=class_ids24,
            zero_division=0
        )
    )

    return precision, recall, f1, support

p1, r1, f1, support = class_metrics24(
    PRED_V1_24
)

p2, r2, f2, _ = class_metrics24(
    PRED_V2_24
)

pe, re, fe, _ = class_metrics24(
    PRED_BEST24
)

CLASS_COMPARISON24 = pd.DataFrame({
    "Class": NAMES24,
    "Support": support,
    "V1 Recall": r1,
    "V2 Recall": r2,
    "Blend Recall": re,
    "Blend Precision": pe,
    "V1 F1": f1,
    "V2 F1": f2,
    "Blend F1": fe
})

print("\nC. ALL TEN ATTACK CLASSES")
print("-" * 85)

print(
    CLASS_COMPARISON24
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 4. Check DoS / Exploits confusion directly.
# ------------------------------------------------------------

dos_id24 = NAMES24.index("DoS")
exp_id24 = NAMES24.index("Exploits")

print("\nD. DoS AND EXPLOITS CONFUSION")
print("-" * 85)

for title, pred in [
    ("V1", PRED_V1_24),
    ("V2", PRED_V2_24),
    ("Blend", PRED_BEST24)
]:

    dos_mask = Y24 == dos_id24
    exp_mask = Y24 == exp_id24

    dos_correct = int(
        np.sum(pred[dos_mask] == dos_id24)
    )

    dos_to_exploits = int(
        np.sum(pred[dos_mask] == exp_id24)
    )

    exploits_correct = int(
        np.sum(pred[exp_mask] == exp_id24)
    )

    exploits_to_dos = int(
        np.sum(pred[exp_mask] == dos_id24)
    )

    print(f"\n{title}")
    print(
        "DoS correctly classified:",
        dos_correct
    )
    print(
        "DoS misclassified as Exploits:",
        dos_to_exploits
    )
    print(
        "Exploits correctly classified:",
        exploits_correct
    )
    print(
        "Exploits misclassified as DoS:",
        exploits_to_dos
    )

# ------------------------------------------------------------
# 5. Compare per-class AP.
#
# AP evaluates probability ranking rather than
# the final argmax decision.
# ------------------------------------------------------------

print("\nE. MACRO AVERAGE PRECISION")
print("-" * 85)

for title, probabilities in [
    ("V1", P_V1),
    ("V2", P_V2),
    ("Blend", P_BEST24)
]:

    ap_scores = [
        average_precision_score(
            (Y24 == i).astype(int),
            probabilities[:, i]
        )
        for i in class_ids24
    ]

    print(
        f"{title:<8}: "
        f"{np.mean(ap_scores):.4f}"
    )

# ------------------------------------------------------------
# 6. Check the 90% target.
# ------------------------------------------------------------

print("\nF. TARGET CHECK")
print("-" * 85)

print(
    "Blend overall accuracy >= 90%:",
    accuracy_score(
        Y24,
        PRED_BEST24
    ) >= 0.90
)

print(
    "Classes with recall >= 90%:",
    int(np.sum(re >= 0.90)),
    "of",
    len(NAMES24)
)

print(
    "Classes still below 90% recall:",
    [
        name
        for name, recall in zip(
            NAMES24, re
        )
        if recall < 0.90
    ]
)

print("\n" + "=" * 85)
print("CELL 24 COMPLETE")
print("=" * 85)

print("New models trained: 0")
print("Existing model weights changed: NO")
print("Official TEST accessed: NO")
print("Saved artifacts modified: NO")
print("Blend permanently saved: NO")

print("=" * 85)


In [ ]:

# ============================================================
# NETGUARD — CURRENT ATTACK DETECTION
# CELL 25 — CONFLICT-AWARE MULTICLASS XGBOOST
#
# Uses development TRAIN labels ONLY to calculate weights.
# Does NOT delete, relabel, or oversample examples.
# Trains exactly ONE experimental model.
#
# NO official TEST access.
# NO Google Drive writes.
# NO modifications to saved models.
# ============================================================

import time
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    average_precision_score
)

print("=" * 85)
print("CELL 25 — CONFLICT-AWARE XGBOOST")
print("=" * 85)

# ------------------------------------------------------------
# 0. Verify previous cells have been executed.
# ------------------------------------------------------------

required25 = [
    "Xtr_processed",
    "Xva_processed",
    "ytr",
    "yva",
    "groups",
    "train_idx",
    "proba",
    "CLASS_NAMES"
]

for name in required25:
    if name not in globals():
        raise RuntimeError(
            f"Missing {name}. Restore the earlier cells first."
        )

classes25 = list(CLASS_NAMES)

y_train25 = np.asarray(ytr, dtype=int)
y_val25 = np.asarray(yva, dtype=int)

group_train25 = (
    groups.iloc[train_idx].to_numpy()
)

assert len(group_train25) == len(y_train25)
assert Xtr_processed.shape[0] == len(y_train25)
assert Xva_processed.shape[0] == len(y_val25)

assert len(classes25) == 10

# ------------------------------------------------------------
# 1. Identify conflicting feature groups in TRAIN ONLY.
# ------------------------------------------------------------

training_audit25 = pd.DataFrame({
    "group": group_train25,
    "label": y_train25
})

group_sizes25 = (
    training_audit25
    .groupby("group")["label"]
    .transform("size")
)

same_label_counts25 = (
    training_audit25
    .groupby(["group", "label"])["label"]
    .transform("size")
)

group_diversity25 = (
    training_audit25
    .groupby("group")["label"]
    .transform("nunique")
)

conflicting_mask25 = (
    group_diversity25 > 1
).to_numpy()

print("\nA. TRAINING GROUP DIAGNOSTICS")
print("-" * 85)

print(
    "Development TRAIN records:",
    len(y_train25)
)

print(
    "Unique feature groups:",
    training_audit25["group"].nunique()
)

print(
    "Records in conflicting groups:",
    int(conflicting_mask25.sum())
)

print(
    "Distinct conflicting groups:",
    training_audit25.loc[
        conflicting_mask25, "group"
    ].nunique()
)

# ------------------------------------------------------------
# 2. Calculate conflict-aware sample weights.
#
# Agreement = number of records with this group's
# label / total records in that feature group.
#
# Unambiguous groups receive initial weight = 1.
# Conflicting groups receive lower weights according
# to their label agreement.
#
# Formula:
#   weight = 0.35 + 0.65 * agreement
#
# All records are retained.
# All original labels are retained.
# ------------------------------------------------------------

agreement25 = (
    same_label_counts25.to_numpy(dtype=float)
    /
    group_sizes25.to_numpy(dtype=float)
)

weights25 = (
    0.35 + 0.65 * agreement25
)

# Normalize mean weight to 1.
weights25 = (
    weights25 / weights25.mean()
)

assert len(weights25) == len(y_train25)
assert np.all(np.isfinite(weights25))
assert np.all(weights25 > 0)

print("\nB. SAMPLE WEIGHT DIAGNOSTICS")
print("-" * 85)

print(
    "Minimum weight:",
    round(float(weights25.min()), 4)
)

print(
    "Maximum weight:",
    round(float(weights25.max()), 4)
)

print(
    "Mean weight:",
    round(float(weights25.mean()), 4)
)

if (~conflicting_mask25).any():

    print(
        "Mean weight (non-conflicting):",
        round(
            float(
                weights25[
                    ~conflicting_mask25
                ].mean()
            ),
            4
        )
    )

if conflicting_mask25.any():

    print(
        "Mean weight (conflicting):",
        round(
            float(
                weights25[
                    conflicting_mask25
                ].mean()
            ),
            4
        )
    )

# ------------------------------------------------------------
# 3. Train ONE conflict-aware XGBoost.
#
# Same hyperparameters as original XGBoost V1.
# Only the training sample weights change.
# ------------------------------------------------------------

model25 = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=len(classes25),

    n_estimators=180,
    max_depth=6,
    learning_rate=0.10,

    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=3,

    tree_method="hist",
    n_jobs=4,
    random_state=42,

    eval_metric="mlogloss"
)

print("\nC. TRAINING")
print("-" * 85)

print("Training one conflict-aware XGBoost...")

start25 = time.time()

model25.fit(
    Xtr_processed,
    y_train25,
    sample_weight=weights25,
    verbose=False
)

training_seconds25 = (
    time.time() - start25
)

print(
    "Training seconds:",
    round(training_seconds25, 2)
)

# ------------------------------------------------------------
# 4. Predict on the unchanged development VALIDATION set.
# ------------------------------------------------------------

p_conflict25 = model25.predict_proba(
    Xva_processed
)

pred_conflict25 = np.argmax(
    p_conflict25,
    axis=1
)

p_v125 = np.asarray(proba)

pred_v125 = np.argmax(
    p_v125,
    axis=1
)

assert p_conflict25.shape == p_v125.shape
assert len(pred_conflict25) == len(y_val25)

# Include V2 if its predictions are still in memory.
pred_v225 = None

if "WEIGHTED_XGB_PROBA22" in globals():

    p_v225 = np.asarray(
        WEIGHTED_XGB_PROBA22
    )

    assert p_v225.shape == p_v125.shape

    pred_v225 = np.argmax(
        p_v225,
        axis=1
    )

# ------------------------------------------------------------
# 5. Overall performance comparison.
# ------------------------------------------------------------

def evaluate25(pred):

    return {
        "Accuracy": accuracy_score(
            y_val25,
            pred
        ),

        "Macro F1": f1_score(
            y_val25,
            pred,
            average="macro",
            zero_division=0
        ),

        "Balanced accuracy": balanced_accuracy_score(
            y_val25,
            pred
        ),

        "Weighted F1": f1_score(
            y_val25,
            pred,
            average="weighted",
            zero_division=0
        )
    }

results25 = [
    {
        "Model": "Original XGBoost V1",
        **evaluate25(pred_v125)
    }
]

if pred_v225 is not None:

    results25.append({
        "Model": "Class-weighted XGBoost V2",
        **evaluate25(pred_v225)
    })

results25.append({
    "Model": "Conflict-aware XGBoost",
    **evaluate25(pred_conflict25)
})

RESULTS25 = pd.DataFrame(
    results25
)

print("\n" + "=" * 85)
print("D. OVERALL PERFORMANCE")
print("=" * 85)

print(
    RESULTS25
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Compare all ten attack classes.
# ------------------------------------------------------------

labels25 = np.arange(
    len(classes25)
)

old_p25, old_r25, old_f25, support25 = (
    precision_recall_fscore_support(
        y_val25,
        pred_v125,
        labels=labels25,
        zero_division=0
    )
)

new_p25, new_r25, new_f25, _ = (
    precision_recall_fscore_support(
        y_val25,
        pred_conflict25,
        labels=labels25,
        zero_division=0
    )
)

comparison25 = pd.DataFrame({
    "Class": classes25,
    "Support": support25,

    "V1 Recall": old_r25,
    "Conflict Recall": new_r25,

    "Conflict Precision": new_p25,

    "V1 F1": old_f25,
    "Conflict F1": new_f25
})

print("\n" + "=" * 85)
print("E. ALL TEN CLASSES")
print("=" * 85)

print(
    comparison25
    .round(4)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 7. Compare average precision (AP).
#
# One-vs-rest AP for each attack class.
# ------------------------------------------------------------

ap_rows25 = []

for i, name in enumerate(classes25):

    class_truth25 = (
        y_val25 == i
    ).astype(int)

    original_ap25 = average_precision_score(
        class_truth25,
        p_v125[:, i]
    )

    conflict_ap25 = average_precision_score(
        class_truth25,
        p_conflict25[:, i]
    )

    ap_rows25.append({
        "Class": name,
        "V1 AP": original_ap25,
        "Conflict AP": conflict_ap25,
        "AP Change": conflict_ap25 - original_ap25
    })

AP_TABLE25 = pd.DataFrame(
    ap_rows25
)

print("\n" + "=" * 85)
print("F. PER-CLASS AVERAGE PRECISION")
print("=" * 85)

print(
    AP_TABLE25
    .round(4)
    .to_string(index=False)
)

print(
    "\nMacro AP:",
    f"V1={AP_TABLE25['V1 AP'].mean():.4f}",
    f"Conflict-aware={AP_TABLE25['Conflict AP'].mean():.4f}"
)

# ------------------------------------------------------------
# 8. Evaluate against our goals.
# ------------------------------------------------------------

metrics25 = evaluate25(
    pred_conflict25
)

baseline_metrics25 = evaluate25(
    pred_v125
)

print("\n" + "=" * 85)
print("G. TARGET CHECK")
print("=" * 85)

print(
    "Overall accuracy >= 90%:",
    metrics25["Accuracy"] >= 0.90
)

print(
    "Accuracy improved vs V1:",
    metrics25["Accuracy"] >
    baseline_metrics25["Accuracy"]
)

print(
    "Macro F1 improved vs V1:",
    metrics25["Macro F1"] >
    baseline_metrics25["Macro F1"]
)

print(
    "Balanced accuracy improved vs V1:",
    metrics25["Balanced accuracy"] >
    baseline_metrics25["Balanced accuracy"]
)

print(
    "Classes with recall >= 90%:",
    int((new_r25 >= 0.90).sum()),
    "of",
    len(classes25)
)

print(
    "Minimum class recall:",
    round(float(new_r25.min()), 4)
)

# ------------------------------------------------------------
# 9. Preserve outputs in Colab memory.
#
# No files are saved or overwritten.
# ------------------------------------------------------------

CONFLICT_XGB_MODEL25 = model25

CONFLICT_XGB_PROBA25 = (
    p_conflict25
)

CONFLICT_XGB_METRICS25 = (
    metrics25
)

CONFLICT_XGB_CLASS_COMPARISON25 = (
    comparison25
)

CONFLICT_XGB_AP_TABLE25 = (
    AP_TABLE25
)

print("\n" + "=" * 85)
print("CELL 25 COMPLETE")
print("=" * 85)

print("New models trained: 1")
print("Training labels changed: NO")
print("Training rows removed: 0")
print("Synthetic samples generated: 0")

print("Official TEST accessed: NO")
print("Previous checkpoints modified: NO")
print("Google Drive files modified: NO")
print("New model saved permanently: NOT YET")

print("=" * 85)


In [ ]:

# ============================================================
# CELL 26 — SAVE CONFLICT-AWARE XGBOOST
#
# No retraining.
# No official TEST access.
# No existing checkpoint overwritten.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import joblib
import numpy as np
import xgboost as xgb

print("=" * 80)
print("CELL 26 — SAVE CONFLICT-AWARE XGBOOST")
print("=" * 80)

required26 = [
    "CONFLICT_XGB_MODEL25",
    "CONFLICT_XGB_PROBA25",
    "CONFLICT_XGB_METRICS25",
    "CURRENT_ATTACK_PREPROCESSOR",
    "CURRENT_ATTACK_FEATURE_COLUMNS",
    "CATEGORICAL_COLS",
    "NUMERICAL_COLS",
    "CLASS_NAMES",
    "Xva",
    "Xva_processed",
    "weights25",
    "conflicting_mask25"
]

for name in required26:
    if name not in globals():
        raise RuntimeError(f"Missing {name}")

# ------------------------------------------------------------
# 1. New experimental directory.
# ------------------------------------------------------------

PROJECT26 = Path(
    "/content/drive/MyDrive/network-attack-forecasting"
)

RUN26 = datetime.now(timezone.utc).strftime(
    "conflict_aware_xgb_%Y%m%d_%H%M%S_%f"
)

SAVE26 = (
    PROJECT26
    / "Current_Attack_Detection"
    / "experiments"
    / RUN26
)

SAVE26.mkdir(parents=True, exist_ok=False)

model_file26 = SAVE26 / "conflict_aware_xgboost.ubj"
prep_file26 = SAVE26 / "preprocessor.joblib"
schema_file26 = SAVE26 / "metadata.json"

# ------------------------------------------------------------
# 2. Save model and preprocessing.
# ------------------------------------------------------------

CONFLICT_XGB_MODEL25.save_model(
    str(model_file26)
)

joblib.dump(
    CURRENT_ATTACK_PREPROCESSOR,
    prep_file26
)

metadata26 = {
    "name": "NetGuard_Conflict_Aware_XGBoost",
    "status": "experimental_not_deployed",

    "dataset": "UNSW-NB15 canonical TRAIN",
    "official_test_used": False,

    "classes": list(CLASS_NAMES),

    "feature_columns": list(
        CURRENT_ATTACK_FEATURE_COLUMNS
    ),

    "categorical_columns": list(
        CATEGORICAL_COLS
    ),

    "numerical_columns": list(
        NUMERICAL_COLS
    ),

    "input_preparation": {
        "categorical": (
            "Convert to pandas string; fill missing "
            "with __MISSING__; convert to str"
        ),
        "numerical": (
            "pd.to_numeric(errors='coerce'); "
            "replace infinities with NaN"
        )
    },

    "training_method": "conflict_aware_sample_weighting",

    "weight_formula": (
        "0.35 + 0.65 * "
        "(count_of_same_label_in_feature_group / "
        "feature_group_size); "
        "then normalized to mean 1"
    ),

    "training_sample_weight_summary": {
        "min": float(np.min(weights25)),
        "max": float(np.max(weights25)),
        "mean": float(np.mean(weights25)),
        "records_in_conflicting_groups": int(
            np.sum(conflicting_mask25)
        )
    },

    "validation_metrics": {
        key: float(value)
        for key, value
        in CONFLICT_XGB_METRICS25.items()
    },

    "evaluation_note": (
        "Exploratory development validation. "
        "Validation data has been examined repeatedly. "
        "An independent final evaluation is required."
    )
}

with schema_file26.open(
    "x",
    encoding="utf-8"
) as f:
    json.dump(metadata26, f, indent=2)

# ------------------------------------------------------------
# 3. Reload and verify both model and preprocessing.
# ------------------------------------------------------------

reloaded_model26 = xgb.XGBClassifier()
reloaded_model26.load_model(str(model_file26))

reloaded_prep26 = joblib.load(prep_file26)

reloaded_features26 = reloaded_prep26.transform(Xva)

# Verify the reloaded preprocessing matches the original.
original_scores26 = CONFLICT_XGB_MODEL25.predict_proba(
    Xva_processed
)

reloaded_scores26 = reloaded_model26.predict_proba(
    reloaded_features26
)

np.testing.assert_allclose(
    original_scores26,
    CONFLICT_XGB_PROBA25,
    rtol=1e-6,
    atol=1e-7
)

np.testing.assert_allclose(
    reloaded_scores26,
    CONFLICT_XGB_PROBA25,
    rtol=1e-6,
    atol=1e-7
)

print("\nModel + preprocessing reload: PASS")

# ------------------------------------------------------------
# 4. Integrity hashes.
# ------------------------------------------------------------

def sha256_26(path):
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):
            digest.update(chunk)

    return digest.hexdigest()

hashes26 = {
    path.name: sha256_26(path)
    for path in [
        model_file26,
        prep_file26,
        schema_file26
    ]
}

with (
    SAVE26 / "artifact_hashes.json"
).open("x", encoding="utf-8") as f:
    json.dump(hashes26, f, indent=2)

print("\nSaved files:")

for path in sorted(SAVE26.iterdir()):
    print(" -", path.name)

print("\n" + "=" * 80)
print("CELL 26 COMPLETE")
print("=" * 80)

print("Saved directory:", SAVE26)
print("Reload verification: PASS")
print("Models trained: 0")
print("Existing checkpoints overwritten: NO")
print("Official TEST accessed: NO")
print("Forecasting models modified: NO")
print("=" * 80)
